In [86]:
# 1 simple test
import pandas as pd
import numpy as np
import os

print("Python is working!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python is working!
Pandas version: 3.0.3
NumPy version: 2.4.4


In [87]:
# 6 load 1 dataset first
labour = pd.read_csv(
    "C:\\Users\\User\\OneDrive - International Islamic University Malaysia\\Documents\\OUTSIDE UNI\\DAX26 UMPSA\\raw_dataset\\pahang_labour_district.csv")

print("\nLabour dataset loaded successfully.")
print("Shape:", labour.shape)
print("\nColumns:")
print(labour.columns.tolist())


Labour dataset loaded successfully.
Shape: (77, 9)

Columns:
['district', 'year', 'labour_force_000', 'employed_000', 'unemployed_000', 'outside_lf_000', 'participation_rate_pct', 'unemployment_rate_pct', 'employment_pop_ratio_pct']


In [88]:
# 7 inspect first 5 rows
print("\nFirst 5 rows:")
print(labour.head())


First 5 rows:
  district  year  labour_force_000  employed_000  unemployed_000  \
0  Bentong  2018              60.5          58.8             1.7   
1  Bentong  2019              60.3          58.4             1.9   
2  Bentong  2020              52.5          50.7             1.8   
3  Bentong  2021              51.5          50.0             1.4   
4  Bentong  2022              52.8          51.8             1.1   

   outside_lf_000  participation_rate_pct  unemployment_rate_pct  \
0            30.8                    66.3                    2.8   
1            31.6                    65.6                    3.2   
2            26.8                    66.2                    3.4   
3            27.5                    65.2                    2.8   
4            28.0                    65.4                    2.1   

   employment_pop_ratio_pct  
0                      64.4  
1                      63.5  
2                      63.9  
3                      63.3  
4                

In [89]:
# 8 Check data types
print("\nData types:")
print(labour.dtypes)


Data types:
district                        str
year                          int64
labour_force_000            float64
employed_000                float64
unemployed_000              float64
outside_lf_000              float64
participation_rate_pct      float64
unemployment_rate_pct       float64
employment_pop_ratio_pct    float64
dtype: object


In [90]:
# 9 check missing values
print("\nMissing values:")
print(labour.isna().sum())


Missing values:
district                    0
year                        0
labour_force_000            0
employed_000                0
unemployed_000              0
outside_lf_000              0
participation_rate_pct      0
unemployment_rate_pct       0
employment_pop_ratio_pct    0
dtype: int64


In [91]:
# 10 Check duplicates
duplicates = labour.duplicated(
    subset=["district", "year"]
).sum()

print("\nDuplicate district-year records:", duplicates)


Duplicate district-year records: 0


In [92]:
# 11 Bringing in crosswalk
crosswalk = pd.read_csv(
    "district_crosswalk.csv"
)

print("\nCrosswalk loaded.")
print("Shape:", crosswalk.shape)
print("\nCrosswalk columns:")
print(crosswalk.columns.tolist())

print("\nFirst 5 crosswalk rows:")
print(crosswalk.head())


Crosswalk loaded.
Shape: (101, 7)

Crosswalk columns:
['source_dataset', 'original_district', 'original_district_clean', 'canonical_district', 'district_id', 'valid_from', 'valid_to']

First 5 crosswalk rows:
  source_dataset  original_district original_district_clean  \
0         income            Bentong                 Bentong   
1         income               Bera                    Bera   
2         income  Cameron Highlands       Cameron Highlands   
3         income           Jerantut                Jerantut   
4         income            Kuantan                 Kuantan   

  canonical_district  district_id  valid_from  valid_to  
0            Bentong          601         NaN       NaN  
1               Bera          611         NaN       NaN  
2  Cameron Highlands          602      2019.0    2024.0  
3           Jerantut          603         NaN       NaN  
4            Kuantan          604         NaN       NaN  


In [93]:
# 12 Check crosswalk before using it
print("\nNumber of unique canonical districts:",
      crosswalk["canonical_district"].nunique())

print("Number of unique district IDs:",
      crosswalk["district_id"].nunique())

print("Missing district IDs:",
      crosswalk["district_id"].isna().sum())


Number of unique canonical districts: 11
Number of unique district IDs: 11
Missing district IDs: 0


In [94]:
# 13 Apply crosswalk to Labour
print(
    labour["district"]
    .dropna()
    .unique()
)

<StringArray>
[          'Bentong',              'Bera', 'Cameron Highlands',
          'Jerantut',           'Kuantan',             'Lipis',
             'Maran',             'Pekan',              'Raub',
            'Rompin',          'Temerloh']
Length: 11, dtype: str


In [95]:
# inspect the crosswalk's source names
print(
    crosswalk["original_district"]
    .dropna()
    .unique()
)

<StringArray>
[          'Bentong',              'Bera', 'Cameron Highlands',
          'Jerantut',           'Kuantan',             'Lipis',
             'Maran',             'Pekan',              'Raub',
            'Rompin',          'Temerloh',  'Cameron Highland',
       'Kuala Lipis']
Length: 13, dtype: str


In [96]:
# 14 merge labour with the crosswalk
labour_clean = labour.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

print("\nLabour after crosswalk:")
print(
    labour_clean[
        [
            "district",
            "canonical_district",
            "district_id"
        ]
    ].drop_duplicates()
)


Labour after crosswalk:
              district canonical_district  district_id
0              Bentong            Bentong          601
63                Bera               Bera          611
126  Cameron Highlands  Cameron Highlands          602
189           Jerantut           Jerantut          603
252            Kuantan            Kuantan          604
315              Lipis              Lipis          605
378              Maran              Maran          610
441              Pekan              Pekan          606
504               Raub               Raub          607
567             Rompin             Rompin          609
630           Temerloh           Temerloh          608


In [97]:
# 15 Important check
unmapped = labour_clean[
    labour_clean["district_id"].isna()
]

print(
    "\nNumber of unmapped Labour rows:",
    len(unmapped)
)

if len(unmapped) > 0:
    print("\nUnmapped district names:")
    print(unmapped["district"].unique())


Number of unmapped Labour rows: 0


In [98]:
# 16 load the rest of the datasets

print("\n" + "="*60)
print("STEP 16: LOADING ALL RAW DATASETS")
print("="*60)

# Folder containing the raw CSV files
raw_path = "C:\\Users\\User\\OneDrive - International Islamic University Malaysia\\Documents\\OUTSIDE UNI\\DAX26 UMPSA\\raw_dataset"


# 1. Income
income = pd.read_csv(
    f"{raw_path}\\pahang_income_district.csv"
)


# 2. Poverty
poverty = pd.read_csv(
    f"{raw_path}\\pahang_poverty_district.csv"
)


# 3. Population
population = pd.read_csv(
    f"{raw_path}\\pahang_population.csv"
)


# 4. Amenities
amenities = pd.read_csv(
    f"{raw_path}\\pahang_amenities.csv"
)


# 5. School enrolment
enrollment = pd.read_csv(
    f"{raw_path}\\pahang_enrollment_school_district.csv"
)


# 6. Inflation / CPI
inflation = pd.read_csv(
    f"{raw_path}\\pahang_inflation_rate.csv"
)


# 7. Crop production 2017–2022
crop_2017_2022 = pd.read_csv(
    f"{raw_path}\\crop_2017_2022.csv"
)


# 8. Crop production 2023
crop_2023 = pd.read_csv(
    f"{raw_path}\\crop_2023.csv"
)


# 9. Agricultural employment
agri_employed = pd.read_csv(
    f"{raw_path}\\agri_employed.csv"
)


print("\nAll raw datasets loaded successfully.")


STEP 16: LOADING ALL RAW DATASETS

All raw datasets loaded successfully.


In [99]:
# 17 Check all DataFrames actually exists
print("\n" + "="*60)
print("DATASET CHECK")
print("="*60)

datasets = {
    "labour": labour,
    "income": income,
    "poverty": poverty,
    "population": population,
    "amenities": amenities,
    "enrollment": enrollment,
    "inflation": inflation,
    "crop_2017_2022": crop_2017_2022,
    "crop_2023": crop_2023,
    "agri_employed": agri_employed
}

for name, df in datasets.items():
    print(
        f"{name:20s} → rows: {df.shape[0]}, columns: {df.shape[1]}"
    )


DATASET CHECK
labour               → rows: 77, columns: 9
income               → rows: 33, columns: 4
poverty              → rows: 33, columns: 4
population           → rows: 26334, columns: 7
amenities            → rows: 48, columns: 6
enrollment           → rows: 936, columns: 6
inflation            → rows: 2772, columns: 5
crop_2017_2022       → rows: 1024, columns: 6
crop_2023            → rows: 12, columns: 11
agri_employed        → rows: 12, columns: 3


In [100]:
# 18 Inspect every DataFrame
print("\n" + "="*70)
print("COLUMN CHECK")
print("="*70)

for name, df in datasets.items():
    
    print("\n" + "-"*70)
    print(name.upper())
    print("-"*70)
    
    print("Columns:")
    print(df.columns.tolist())
    
    print("\nFirst 3 rows:")
    print(df.head(3))


COLUMN CHECK

----------------------------------------------------------------------
LABOUR
----------------------------------------------------------------------
Columns:
['district', 'year', 'labour_force_000', 'employed_000', 'unemployed_000', 'outside_lf_000', 'participation_rate_pct', 'unemployment_rate_pct', 'employment_pop_ratio_pct']

First 3 rows:
  district  year  labour_force_000  employed_000  unemployed_000  \
0  Bentong  2018              60.5          58.8             1.7   
1  Bentong  2019              60.3          58.4             1.9   
2  Bentong  2020              52.5          50.7             1.8   

   outside_lf_000  participation_rate_pct  unemployment_rate_pct  \
0            30.8                    66.3                    2.8   
1            31.6                    65.6                    3.2   
2            26.8                    66.2                    3.4   

   employment_pop_ratio_pct  
0                      64.4  
1                      63.5  
2   

In [101]:
# 19 Create cleaned DataFrames

# income
income_clean = income.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# poverty
poverty_clean = poverty.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# population
population_clean = population.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# amenities
amenities_clean = amenities.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# enrolment
enrollment_clean = enrollment.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# crop 2017
crop_clean = crop_2017_2022.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# crop 2023
crop_2023_clean = crop_2023.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# agriculture employment 2023
agri_clean = agri_employed.merge(
    crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

# inflation - state level
inflation_clean = inflation.copy()


In [102]:
# 20 check all cleaned DataFrames
cleaned_datasets = {
    "labour_clean": labour_clean,
    "income_clean": income_clean,
    "poverty_clean": poverty_clean,
    "population_clean": population_clean,
    "amenities_clean": amenities_clean,
    "enrollment_clean": enrollment_clean,
    "crop_clean": crop_clean,
    "crop_2023_clean": crop_2023_clean,
    "agri_clean": agri_clean,
    "inflation_clean": inflation_clean
}

print("\n" + "="*70)
print("CLEANED DATASET CHECK")
print("="*70)

for name, df in cleaned_datasets.items():
    print(
        f"{name:20s} → rows: {df.shape[0]}, columns: {df.shape[1]}"
    )


CLEANED DATASET CHECK
labour_clean         → rows: 693, columns: 12
income_clean         → rows: 297, columns: 7
poverty_clean        → rows: 297, columns: 7
population_clean     → rows: 224238, columns: 10
amenities_clean      → rows: 400, columns: 9
enrollment_clean     → rows: 7440, columns: 9
crop_clean           → rows: 9216, columns: 9
crop_2023_clean      → rows: 100, columns: 14
agri_clean           → rows: 100, columns: 6
inflation_clean      → rows: 2772, columns: 5


In [103]:
# 21 check unmapped districts
print("\n" + "="*70)
print("CROSSWALK VALIDATION")
print("="*70)

for name, df in cleaned_datasets.items():
    
    if "district_id" in df.columns:
        
        unmapped = df["district_id"].isna().sum()
        
        print(
            f"{name:20s} → unmapped rows: {unmapped}"
        )


CROSSWALK VALIDATION
labour_clean         → unmapped rows: 0
income_clean         → unmapped rows: 0
poverty_clean        → unmapped rows: 0
population_clean     → unmapped rows: 0
amenities_clean      → unmapped rows: 4
enrollment_clean     → unmapped rows: 78
crop_clean           → unmapped rows: 0
crop_2023_clean      → unmapped rows: 1
agri_clean           → unmapped rows: 1


In [104]:
# 22 Identify the unmapped district names
# ============================================================
# STEP 22 — IDENTIFY UNMAPPED DISTRICTS
# ============================================================

print("\n" + "="*70)
print("STEP 22: INVESTIGATING UNMAPPED DISTRICTS")
print("="*70)


# ------------------------------------------------------------
# Amenities
# ------------------------------------------------------------

print("\n[1] AMENITIES")

unmapped_amenities = amenities_clean[
    amenities_clean["district_id"].isna()
]

print(
    "Number of unmapped rows:",
    len(unmapped_amenities)
)

print(
    "Unmapped district values:"
)

print(
    unmapped_amenities["district"].value_counts()
)


# ------------------------------------------------------------
# Enrolment
# ------------------------------------------------------------

print("\n[2] ENROLMENT")

unmapped_enrollment = enrollment_clean[
    enrollment_clean["district_id"].isna()
]

print(
    "Number of unmapped rows:",
    len(unmapped_enrollment)
)

print(
    "Unmapped district values:"
)

print(
    unmapped_enrollment["district"].value_counts()
)


# ------------------------------------------------------------
# Crop 2023
# ------------------------------------------------------------

print("\n[3] CROP 2023")

unmapped_crop_2023 = crop_2023_clean[
    crop_2023_clean["district_id"].isna()
]

print(
    "Number of unmapped rows:",
    len(unmapped_crop_2023)
)

print(
    "Unmapped district values:"
)

print(
    unmapped_crop_2023["district"].value_counts()
)


# ------------------------------------------------------------
# Agricultural employment
# ------------------------------------------------------------

print("\n[4] AGRICULTURAL EMPLOYMENT")

unmapped_agri = agri_clean[
    agri_clean["district_id"].isna()
]

print(
    "Number of unmapped rows:",
    len(unmapped_agri)
)

print(
    "Unmapped district values:"
)

print(
    unmapped_agri["district"].value_counts()
)


STEP 22: INVESTIGATING UNMAPPED DISTRICTS

[1] AMENITIES
Number of unmapped rows: 4
Unmapped district values:
district
All Districts    4
Name: count, dtype: int64

[2] ENROLMENT
Number of unmapped rows: 78
Unmapped district values:
district
All Districts    78
Name: count, dtype: int64

[3] CROP 2023
Number of unmapped rows: 1
Unmapped district values:
district
Total    1
Name: count, dtype: int64

[4] AGRICULTURAL EMPLOYMENT
Number of unmapped rows: 1
Unmapped district values:
district
Total    1
Name: count, dtype: int64


In [105]:
# 23 showing complete rows for unmapped dataset
print("\n" + "="*70)
print("UNMAPPED AMENITIES ROWS")
print("="*70)

print(unmapped_amenities)


UNMAPPED AMENITIES ROWS
         date   state       district  piped_water_%  sanitation_%  \
0  2016-01-01  Pahang  All Districts           97.8         99.75   
1  2019-01-01  Pahang  All Districts           97.9         99.98   
2  2022-01-01  Pahang  All Districts           98.3         99.40   
3  2024-01-01  Pahang  All Districts            NaN           NaN   

   electricity_% original_district canonical_district  district_id  
0         100.00               NaN                NaN          NaN  
1         100.00               NaN                NaN          NaN  
2          99.51               NaN                NaN          NaN  
3         100.00               NaN                NaN          NaN  


In [106]:
print("\n" + "="*70)
print("UNMAPPED ENROLMENT ROWS")
print("="*70)

print(unmapped_enrollment)


UNMAPPED ENROLMENT ROWS
     state       district           stage   sex        date  students  \
0   Pahang  All Districts         primary  both  2017-01-01    145117   
1   Pahang  All Districts         primary  both  2018-01-01    145698   
2   Pahang  All Districts         primary  both  2019-01-01    146680   
3   Pahang  All Districts         primary  both  2020-01-01    147764   
4   Pahang  All Districts         primary  both  2021-01-01    150671   
..     ...            ...             ...   ...         ...       ...   
73  Pahang  All Districts  post_secondary  male  2021-01-01      1546   
74  Pahang  All Districts  post_secondary  male  2022-01-01      1766   
75  Pahang  All Districts  post_secondary  male  2023-06-30      2084   
76  Pahang  All Districts  post_secondary  male  2024-06-30      1861   
77  Pahang  All Districts  post_secondary  male  2025-06-30      2822   

   original_district canonical_district  district_id  
0                NaN                NaN    

In [107]:
print("\n" + "="*70)
print("UNMAPPED CROP 2023 ROWS")
print("="*70)

print(unmapped_crop_2023)


UNMAPPED CROP 2023 ROWS
   district paddy_production_tonnes oil_palm_production_tonnes  \
99    Total                 55650.2                 15492947.6   

    rubber_production_tonnes  fruits_production_tonnes  \
99                   49666.9                  241742.6   

    vegetable_production_tonnes kenaf_production_tonnes  \
99                      70.8362                  5224.2   

    cocoa_production_tonnes  pepper_production_tonnes  \
99                  70.8362                   70.8362   

    pineapple_production_tonnes  others_production_tonnes original_district  \
99                      70.8362                   70.8362               NaN   

   canonical_district  district_id  
99                NaN          NaN  


In [108]:
print("\n" + "="*70)
print("UNMAPPED AGRICULTURAL EMPLOYMENT ROW")
print("="*70)

print(unmapped_agri)


UNMAPPED AGRICULTURAL EMPLOYMENT ROW
   district total_employment avg_annual_salary_rm original_district  \
99    Total          193,216               20,987               NaN   

   canonical_district  district_id  
99                NaN          NaN  


In [109]:
# compare unmapped names against crosswalk
print("\n" + "="*70)
print("CROSSWALK DISTRICT NAMES")
print("="*70)

print(
    sorted(
        crosswalk["original_district"]
        .dropna()
        .astype(str)
        .unique()
    )
)


CROSSWALK DISTRICT NAMES
['Bentong', 'Bera', 'Cameron Highland', 'Cameron Highlands', 'Jerantut', 'Kuala Lipis', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']


In [110]:
print("\n" + "="*70)
print("CANONICAL DISTRICTS")
print("="*70)

print(
    sorted(
        crosswalk["canonical_district"]
        .dropna()
        .astype(str)
        .unique()
    )
)


CANONICAL DISTRICTS
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']


In [111]:
# 24 remove aggregate rows

# amenities
amenities_clean = amenities_clean[
    amenities_clean["district"].ne("All Districts")
].copy()

print(
    "Amenities rows after removing 'All Districts':",
    len(amenities_clean)
)

# enrolment
enrollment_clean = enrollment_clean[
    enrollment_clean["district"].ne("All Districts")
].copy()

print(
    "Enrolment rows after removing 'All Districts':",
    len(enrollment_clean)
)

# crop 2023
crop_2023_clean = crop_2023_clean[
    crop_2023_clean["district"].ne("Total")
].copy()

print(
    "Crop 2023 rows after removing 'Total':",
    len(crop_2023_clean)
)

# agricultural employment
agri_clean = agri_clean[
    agri_clean["district"].ne("Total")
].copy()

print(
    "Agricultural employment rows after removing 'Total':",
    len(agri_clean)
)


Amenities rows after removing 'All Districts': 396
Enrolment rows after removing 'All Districts': 7362
Crop 2023 rows after removing 'Total': 99
Agricultural employment rows after removing 'Total': 99


In [112]:
print("\n" + "="*70)
print("CROSSWALK VALIDATION — AFTER REMOVING AGGREGATES")
print("="*70)

cleaned_datasets = {
    "labour_clean": labour_clean,
    "income_clean": income_clean,
    "poverty_clean": poverty_clean,
    "population_clean": population_clean,
    "amenities_clean": amenities_clean,
    "enrollment_clean": enrollment_clean,
    "crop_clean": crop_clean,
    "crop_2023_clean": crop_2023_clean,
    "agri_clean": agri_clean
}

for name, df in cleaned_datasets.items():

    unmapped = df["district_id"].isna().sum()

    print(
        f"{name:20s} → unmapped rows: {unmapped}"
    )


CROSSWALK VALIDATION — AFTER REMOVING AGGREGATES
labour_clean         → unmapped rows: 0
income_clean         → unmapped rows: 0
poverty_clean        → unmapped rows: 0
population_clean     → unmapped rows: 0
amenities_clean      → unmapped rows: 0
enrollment_clean     → unmapped rows: 0
crop_clean           → unmapped rows: 0
crop_2023_clean      → unmapped rows: 0
agri_clean           → unmapped rows: 0


In [113]:
# 28 check crosswalk for duplicate mappings
print("\n" + "="*70)
print("CHECKING CROSSWALK FOR DUPLICATE DISTRICT MAPPINGS")
print("="*70)

crosswalk_duplicates = (
    crosswalk
    .groupby("original_district")
    .size()
    .sort_values(ascending=False)
)

print(crosswalk_duplicates)


CHECKING CROSSWALK FOR DUPLICATE DISTRICT MAPPINGS
original_district
Bentong              9
Bera                 9
Cameron Highlands    9
Jerantut             9
Kuantan              9
Rompin               9
Lipis                9
Maran                9
Pekan                9
Temerloh             9
Raub                 9
Cameron Highland     1
Kuala Lipis          1
dtype: int64


In [114]:
# 29 check exactly how many crosswalk rows
print("\nCrosswalk shape:")
print(crosswalk.shape)

print("\nNumber of rows:")
print(len(crosswalk))

print("\nUnique original districts:")
print(crosswalk["original_district"].nunique())

print("\nUnique district IDs:")
print(crosswalk["district_id"].nunique())


Crosswalk shape:
(101, 7)

Number of rows:
101

Unique original districts:
13

Unique district IDs:
11


In [115]:
# 30 check whether ori raw datasets still intact
print("\n" + "="*70)
print("RAW DATASET ROW COUNTS")
print("="*70)

print("Amenities:", len(amenities))
print("Enrolment:", len(enrollment))
print("Crop 2023:", len(crop_2023))
print("Agricultural employment:", len(agri_employed))


RAW DATASET ROW COUNTS
Amenities: 48
Enrolment: 936
Crop 2023: 12
Agricultural employment: 12


In [116]:
# 31 check current cleaned rows
print("\n" + "="*70)
print("CURRENT CLEANED DATASET ROW COUNTS")
print("="*70)

print("Amenities clean:", len(amenities_clean))
print("Enrolment clean:", len(enrollment_clean))
print("Crop 2023 clean:", len(crop_2023_clean))
print("Agricultural employment clean:", len(agri_clean))


CURRENT CLEANED DATASET ROW COUNTS
Amenities clean: 396
Enrolment clean: 7362
Crop 2023 clean: 99
Agricultural employment clean: 99


In [117]:
# 33 inspect source_dataset values
print("\n" + "="*70)
print("SOURCE DATASETS IN CROSSWALK")
print("="*70)

print(
    crosswalk["source_dataset"]
    .value_counts()
)


SOURCE DATASETS IN CROSSWALK
source_dataset
population        12
enrolment         12
income            11
poverty           11
lfs               11
amenities         11
crop_2023         11
crop_2017_2022    11
agri_employed     11
Name: count, dtype: int64


In [118]:
# 34 check crosswalk structure
print(
    crosswalk[
        [
            "source_dataset",
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ].to_string(index=False)
)

source_dataset original_district canonical_district  district_id
        income           Bentong            Bentong          601
        income              Bera               Bera          611
        income Cameron Highlands  Cameron Highlands          602
        income          Jerantut           Jerantut          603
        income           Kuantan            Kuantan          604
        income             Lipis              Lipis          605
        income             Maran              Maran          610
        income             Pekan              Pekan          606
        income              Raub               Raub          607
        income            Rompin             Rompin          609
        income          Temerloh           Temerloh          608
       poverty           Bentong            Bentong          601
       poverty              Bera               Bera          611
       poverty Cameron Highlands  Cameron Highlands          602
       poverty          J

In [119]:
# 36 select lfs crosswalk
lfs_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "lfs"
].copy()

print("\n" + "="*70)
print("LFS CROSSWALK")
print("="*70)

print(
    lfs_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ].to_string(index=False)
)

print("\nNumber of LFS crosswalk rows:", len(lfs_crosswalk))


LFS CROSSWALK
original_district canonical_district  district_id
          Bentong            Bentong          601
             Bera               Bera          611
Cameron Highlands  Cameron Highlands          602
         Jerantut           Jerantut          603
          Kuantan            Kuantan          604
            Lipis              Lipis          605
            Maran              Maran          610
            Pekan              Pekan          606
             Raub               Raub          607
           Rompin             Rompin          609
         Temerloh           Temerloh          608

Number of LFS crosswalk rows: 11


In [120]:
# 37 test the labour merge again
labour_test = labour.merge(
    lfs_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)

print("\nOriginal Labour rows:")
print(len(labour))

print("\nLabour rows after crosswalk:")
print(len(labour_test))

print("\nUnmapped Labour rows:")
print(
    labour_test["district_id"].isna().sum()
)


Original Labour rows:
77

Labour rows after crosswalk:
77

Unmapped Labour rows:
0


In [121]:
# 38 check each district maps exactly once
print("\n" + "="*70)
print("CHECK LFS CROSSWALK UNIQUENESS")
print("="*70)

print(
    lfs_crosswalk
    .groupby("original_district")
    .size()
)


CHECK LFS CROSSWALK UNIQUENESS
original_district
Bentong              1
Bera                 1
Cameron Highlands    1
Jerantut             1
Kuantan              1
Lipis                1
Maran                1
Pekan                1
Raub                 1
Rompin               1
Temerloh             1
dtype: int64


In [122]:
# 39 Rebuild all cleaned DataFrames correctly
print("\n" + "="*70)
print("STEP 39: REBUILDING CLEANED DATASETS")
print("="*70)


# ------------------------------------------------------------
# 1. LABOUR
# Crosswalk source_dataset = lfs
# ------------------------------------------------------------

lfs_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "lfs"
].copy()

labour_clean = labour.merge(
    lfs_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 2. INCOME
# Crosswalk source_dataset = income
# ------------------------------------------------------------

income_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "income"
].copy()

income_clean = income.merge(
    income_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 3. POVERTY
# Crosswalk source_dataset = poverty
# ------------------------------------------------------------

poverty_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "poverty"
].copy()

poverty_clean = poverty.merge(
    poverty_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 4. POPULATION
# Crosswalk source_dataset = population
# ------------------------------------------------------------

population_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "population"
].copy()

population_clean = population.merge(
    population_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 5. AMENITIES
# Crosswalk source_dataset = amenities
# ------------------------------------------------------------

amenities_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "amenities"
].copy()

amenities_clean = amenities.merge(
    amenities_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 6. ENROLMENT
# Crosswalk source_dataset = enrolment
# ------------------------------------------------------------

enrollment_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "enrolment"
].copy()

enrollment_clean = enrollment.merge(
    enrollment_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 7. CROP 2017–2022
# Crosswalk source_dataset = crop_2017_2022
# ------------------------------------------------------------

crop_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "crop_2017_2022"
].copy()

crop_clean = crop_2017_2022.merge(
    crop_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 8. CROP 2023
# Crosswalk source_dataset = crop_2023
# ------------------------------------------------------------

crop_2023_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "crop_2023"
].copy()

crop_2023_clean = crop_2023.merge(
    crop_2023_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 9. AGRICULTURAL EMPLOYMENT
# Crosswalk source_dataset = agri_employed
# ------------------------------------------------------------

agri_crosswalk = crosswalk[
    crosswalk["source_dataset"] == "agri_employed"
].copy()

agri_clean = agri_employed.merge(
    agri_crosswalk[
        [
            "original_district",
            "canonical_district",
            "district_id"
        ]
    ],
    left_on="district",
    right_on="original_district",
    how="left"
)


# ------------------------------------------------------------
# 10. INFLATION / CPI
# State-level data → no district crosswalk
# ------------------------------------------------------------

inflation_clean = inflation.copy()


print("\nAll cleaned DataFrames rebuilt.")


STEP 39: REBUILDING CLEANED DATASETS



All cleaned DataFrames rebuilt.


In [123]:
# 40 check row counts
print("\n" + "="*70)
print("ROW COUNT CHECK")
print("="*70)

print(f"Labour:                 {len(labour)} → {len(labour_clean)}")
print(f"Income:                 {len(income)} → {len(income_clean)}")
print(f"Poverty:                {len(poverty)} → {len(poverty_clean)}")
print(f"Population:             {len(population)} → {len(population_clean)}")
print(f"Amenities:              {len(amenities)} → {len(amenities_clean)}")
print(f"Enrolment:              {len(enrollment)} → {len(enrollment_clean)}")
print(f"Crop 2017–2022:         {len(crop_2017_2022)} → {len(crop_clean)}")
print(f"Crop 2023:              {len(crop_2023)} → {len(crop_2023_clean)}")
print(f"Agricultural employment:{len(agri_employed)} → {len(agri_clean)}")
print(f"Inflation:              {len(inflation)} → {len(inflation_clean)}")


ROW COUNT CHECK
Labour:                 77 → 77
Income:                 33 → 33
Poverty:                33 → 33
Population:             26334 → 26334
Amenities:              48 → 48
Enrolment:              936 → 936
Crop 2017–2022:         1024 → 1024
Crop 2023:              12 → 12
Agricultural employment:12 → 12
Inflation:              2772 → 2772


In [124]:
# 41 check unmapped rows
print("\n" + "="*70)
print("UNMAPPED DISTRICT CHECK")
print("="*70)

cleaned_datasets = {
    "labour_clean": labour_clean,
    "income_clean": income_clean,
    "poverty_clean": poverty_clean,
    "population_clean": population_clean,
    "amenities_clean": amenities_clean,
    "enrollment_clean": enrollment_clean,
    "crop_clean": crop_clean,
    "crop_2023_clean": crop_2023_clean,
    "agri_clean": agri_clean
}

for name, df in cleaned_datasets.items():

    unmapped = df["district_id"].isna().sum()

    print(
        f"{name:20s} → {unmapped} unmapped rows"
    )


UNMAPPED DISTRICT CHECK
labour_clean         → 0 unmapped rows
income_clean         → 0 unmapped rows
poverty_clean        → 0 unmapped rows
population_clean     → 0 unmapped rows
amenities_clean      → 4 unmapped rows
enrollment_clean     → 78 unmapped rows
crop_clean           → 0 unmapped rows
crop_2023_clean      → 1 unmapped rows
agri_clean           → 1 unmapped rows


In [125]:
# 43 check the unique 11 district IDs
print("\n" + "="*70)
print("UNIQUE DISTRICT CHECK")
print("="*70)

for name, df in cleaned_datasets.items():

    print(
        f"{name:20s} → "
        f"{df['district_id'].nunique()} unique district IDs"
    )


UNIQUE DISTRICT CHECK
labour_clean         → 11 unique district IDs
income_clean         → 11 unique district IDs
poverty_clean        → 11 unique district IDs
population_clean     → 11 unique district IDs
amenities_clean      → 11 unique district IDs
enrollment_clean     → 11 unique district IDs
crop_clean           → 11 unique district IDs
crop_2023_clean      → 11 unique district IDs
agri_clean           → 11 unique district IDs


In [126]:
# 44 verify remaining unmapped rows
print("\n" + "="*70)
print("STEP 44: VERIFY REMAINING UNMAPPED ROWS")
print("="*70)


# Amenities
print("\nAMENITIES:")
print(
    amenities_clean.loc[
        amenities_clean["district_id"].isna(),
        ["district", "canonical_district", "district_id"]
    ].drop_duplicates()
)


# Enrolment
print("\nENROLMENT:")
print(
    enrollment_clean.loc[
        enrollment_clean["district_id"].isna(),
        ["district", "canonical_district", "district_id"]
    ].drop_duplicates()
)


# Crop 2023
print("\nCROP 2023:")
print(
    crop_2023_clean.loc[
        crop_2023_clean["district_id"].isna(),
        ["district", "canonical_district", "district_id"]
    ].drop_duplicates()
)


# Agricultural employment
print("\nAGRICULTURAL EMPLOYMENT:")
print(
    agri_clean.loc[
        agri_clean["district_id"].isna(),
        ["district", "canonical_district", "district_id"]
    ].drop_duplicates()
)


STEP 44: VERIFY REMAINING UNMAPPED ROWS

AMENITIES:
        district canonical_district  district_id
0  All Districts                NaN          NaN

ENROLMENT:
        district canonical_district  district_id
0  All Districts                NaN          NaN

CROP 2023:
   district canonical_district  district_id
11    Total                NaN          NaN

AGRICULTURAL EMPLOYMENT:
   district canonical_district  district_id
11    Total                NaN          NaN


In [127]:
# 45 remove only those aggregate rows 

# amenities
amenities_clean = amenities_clean[
    amenities_clean["district"] != "All Districts"
].copy()

# enrolment
enrollment_clean = enrollment_clean[
    enrollment_clean["district"] != "All Districts"
].copy()

# crop 2023
crop_2023_clean = crop_2023_clean[
    crop_2023_clean["district"] != "Total"
].copy()

# agricultural employment
agri_clean = agri_clean[
    agri_clean["district"] != "Total"
].copy()

In [128]:
# 46 Check row counts
print("\n" + "="*70)
print("ROW COUNTS AFTER REMOVING AGGREGATES")
print("="*70)

print("Labour:", len(labour_clean))
print("Income:", len(income_clean))
print("Poverty:", len(poverty_clean))
print("Population:", len(population_clean))
print("Amenities:", len(amenities_clean))
print("Enrolment:", len(enrollment_clean))
print("Crop 2017–2022:", len(crop_clean))
print("Crop 2023:", len(crop_2023_clean))
print("Agricultural employment:", len(agri_clean))


ROW COUNTS AFTER REMOVING AGGREGATES
Labour: 77
Income: 33
Poverty: 33
Population: 26334
Amenities: 44
Enrolment: 858
Crop 2017–2022: 1024
Crop 2023: 11
Agricultural employment: 11


In [129]:
# 47 final unmapped check
print("\n" + "="*70)
print("FINAL DISTRICT MAPPING CHECK")
print("="*70)

cleaned_datasets = {
    "labour_clean": labour_clean,
    "income_clean": income_clean,
    "poverty_clean": poverty_clean,
    "population_clean": population_clean,
    "amenities_clean": amenities_clean,
    "enrollment_clean": enrollment_clean,
    "crop_clean": crop_clean,
    "crop_2023_clean": crop_2023_clean,
    "agri_clean": agri_clean
}

for name, df in cleaned_datasets.items():

    print(
        f"{name:20s} → "
        f"unmapped: {df['district_id'].isna().sum()}"
    )


FINAL DISTRICT MAPPING CHECK
labour_clean         → unmapped: 0
income_clean         → unmapped: 0
poverty_clean        → unmapped: 0
population_clean     → unmapped: 0
amenities_clean      → unmapped: 0
enrollment_clean     → unmapped: 0
crop_clean           → unmapped: 0
crop_2023_clean      → unmapped: 0
agri_clean           → unmapped: 0


In [130]:
# 48 Final check on number of district
print("\n" + "="*70)
print("STEP 48: NUMBER OF DISTRICTS")
print("="*70)

for name, df in cleaned_datasets.items():

    number_of_districts = df["district_id"].nunique()

    print(
        f"{name:20s} → {number_of_districts} districts"
    )


STEP 48: NUMBER OF DISTRICTS
labour_clean         → 11 districts
income_clean         → 11 districts
poverty_clean        → 11 districts
population_clean     → 11 districts
amenities_clean      → 11 districts
enrollment_clean     → 11 districts
crop_clean           → 11 districts
crop_2023_clean      → 11 districts
agri_clean           → 11 districts


In [131]:
# 49 Standardised time/year variables
# Labour
print("\n" + "="*70)
print("STEP 49A — LABOUR YEAR CHECK")
print("="*70)

print("Year data type:")
print(labour_clean["year"].dtype)

print("\nUnique years:")
print(sorted(labour_clean["year"].unique()))

# to make it explicitly numeric
labour_clean["year"] = pd.to_numeric(
    labour_clean["year"],
    errors="coerce"
).astype("Int64")
print(labour_clean["year"].dtype)


STEP 49A — LABOUR YEAR CHECK
Year data type:
int64

Unique years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Int64


In [132]:
# income
print("\n" + "="*70)
print("STEP 49B — INCOME YEAR CHECK")
print("="*70)

print("Year data type:")
print(income_clean["year"].dtype)

print("\nUnique years:")
print(sorted(income_clean["year"].unique()))

# standardised
income_clean["year"] = pd.to_numeric(
    income_clean["year"],
    errors="coerce"
).astype("Int64")

# poverty
print("\n" + "="*70)
print("STEP 49C — POVERTY YEAR CHECK")
print("="*70)

print("Year data type:")
print(poverty_clean["year"].dtype)

print("\nUnique years:")
print(sorted(poverty_clean["year"].unique()))
poverty_clean["year"] = pd.to_numeric(
    poverty_clean["year"],
    errors="coerce"
).astype("Int64")

# population
print("\n" + "="*70)
print("STEP 49D — POPULATION DATE CHECK")
print("="*70)

print("Date data type:")
print(population_clean["date"].dtype)

print("\nFirst 10 dates:")
print(
    population_clean["date"]
    .head(10)
)

# convert 
population_clean["date"] = pd.to_datetime(
    population_clean["date"],
    errors="coerce"
)

# extract the year
population_clean["year"] = (
    population_clean["date"].dt.year
)

print("\nPopulation years:")
print(
    sorted(
        population_clean["year"]
        .dropna()
        .unique()
    )
)

# check population invalid dates
print(
    "\nMissing population years:",
    population_clean["year"].isna().sum()
)

# amenities
print("\n" + "="*70)
print("STEP 49F — AMENITIES DATE CHECK")
print("="*70)

print("Date data type:")
print(amenities_clean["date"].dtype)

print("\nFirst 10 dates:")
print(
    amenities_clean["date"].head(10)
)

# convert
amenities_clean["date"] = pd.to_datetime(
    amenities_clean["date"],
    errors="coerce"
)

# extract year
amenities_clean["year"] = (
    amenities_clean["date"].dt.year
)

# check
print("\nAmenities years:")
print(
    sorted(
        amenities_clean["year"]
        .dropna()
        .unique()
    )
)

print(
    "Missing amenities years:",
    amenities_clean["year"].isna().sum()
)

# enrolment
print("\n" + "="*70)
print("STEP 49G — ENROLMENT DATE CHECK")
print("="*70)

print("Date data type:")
print(enrollment_clean["date"].dtype)

print("\nFirst 10 dates:")
print(
    enrollment_clean["date"].head(10)
)

# convert
enrollment_clean["date"] = pd.to_datetime(
    enrollment_clean["date"],
    errors="coerce"
)

# extract year
enrollment_clean["year"] = (
    enrollment_clean["date"].dt.year
)

print("\nEnrolment years:")
print(
    sorted(
        enrollment_clean["year"]
        .dropna()
        .unique()
    )
)

print(
    "Missing enrolment years:",
    enrollment_clean["year"].isna().sum()
)

# Crop_2017
print("\n" + "="*70)
print("STEP 49H — CROP DATE CHECK")
print("="*70)

print("Date data type:")
print(crop_clean["date"].dtype)

print("\nFirst 10 dates:")
print(
    crop_clean["date"].head(10)
)

# convert
crop_clean["date"] = pd.to_datetime(
    crop_clean["date"],
    errors="coerce"
)

# extract year
crop_clean["year"] = (
    crop_clean["date"].dt.year
)

print("\nCrop years:")
print(
    sorted(
        crop_clean["year"]
        .dropna()
        .unique()
    )
)

print(
    "Missing crop years:",
    crop_clean["year"].isna().sum()
)

# crop 2023
crop_2023_clean["year"] = 2023

print("\nCrop 2023 years:")
print(
    crop_2023_clean["year"].unique()
)

# agricultural employment - does not have a year variable

# inflation
print("\n" + "="*70)
print("STEP 49K — INFLATION DATE CHECK")
print("="*70)

print(
    inflation_clean.columns.tolist()
)

print(
    inflation_clean["date_by_month"].head(10)
)

# convert
inflation_clean["date_by_month"] = pd.to_datetime(
    inflation_clean["date_by_month"],
    errors="coerce"
)

# create year
inflation_clean["year"] = (
    inflation_clean["date_by_month"].dt.year
)


STEP 49B — INCOME YEAR CHECK
Year data type:
int64

Unique years:
[np.int64(2019), np.int64(2022), np.int64(2024)]

STEP 49C — POVERTY YEAR CHECK
Year data type:
int64

Unique years:
[np.int64(2019), np.int64(2022), np.int64(2024)]

STEP 49D — POPULATION DATE CHECK
Date data type:
str

First 10 dates:
0    2020-01-01
1    2020-01-01
2    2020-01-01
3    2020-01-01
4    2020-01-01
5    2020-01-01
6    2020-01-01
7    2020-01-01
8    2020-01-01
9    2020-01-01
Name: date, dtype: str



Population years:
[np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

Missing population years: 0

STEP 49F — AMENITIES DATE CHECK
Date data type:
str

First 10 dates:
4     2016-01-01
5     2019-01-01
6     2022-01-01
7     2024-01-01
8     2016-01-01
9     2019-01-01
10    2022-01-01
11    2024-01-01
12    2016-01-01
13    2019-01-01
Name: date, dtype: str

Amenities years:
[np.int32(2016), np.int32(2019), np.int32(2022), np.int32(2024)]
Missing amenities years: 0

STEP 49G — ENROLMENT DATE CHECK
Date data type:
str

First 10 dates:
78    2017-01-01
79    2018-01-01
80    2019-01-01
81    2020-01-01
82    2021-01-01
83    2022-01-01
84    2023-06-30
85    2024-06-30
86    2025-06-30
87    2017-01-01
Name: date, dtype: str

Enrolment years:
[np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Missing enrolment years: 0

STEP 49H — CROP DATE CHECK
D

In [133]:
# final year check
print("\n" + "="*70)
print("STEP 49L — FINAL YEAR CHECK")
print("="*70)

print(
    "Labour:",
    sorted(labour_clean["year"].dropna().unique())
)

print(
    "Income:",
    sorted(income_clean["year"].dropna().unique())
)

print(
    "Poverty:",
    sorted(poverty_clean["year"].dropna().unique())
)

print(
    "Population:",
    sorted(population_clean["year"].dropna().unique())
)

print(
    "Amenities:",
    sorted(amenities_clean["year"].dropna().unique())
)

print(
    "Enrolment:",
    sorted(enrollment_clean["year"].dropna().unique())
)

print(
    "Crop:",
    sorted(crop_clean["year"].dropna().unique())
)

print(
    "Crop 2023:",
    sorted(crop_2023_clean["year"].dropna().unique())
)

print(
    "Inflation:",
    sorted(inflation_clean["year"].dropna().unique())
)


STEP 49L — FINAL YEAR CHECK
Labour: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Income: [np.int64(2019), np.int64(2022), np.int64(2024)]
Poverty: [np.int64(2019), np.int64(2022), np.int64(2024)]
Population: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Amenities: [np.int32(2016), np.int32(2019), np.int32(2022), np.int32(2024)]
Enrolment: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Crop: [np.int32(2017)]
Crop 2023: [np.int64(2023)]
Inflation: [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


In [134]:
# assign year to agricultural employment
agri_clean["year"] = 2023

print("\nAgricultural employment year:")
print(
    agri_clean["year"].unique()
)
# Agricultural employment dataset represents 2023.
# The original CSV does not contain a year column,
# so 2023 is assigned during preprocessing based on
# the known dataset reference/year.


Agricultural employment year:
[2023]


# 50 Numeric-variable cleaning

LABOUR DATASET

In [135]:
# 50A Labour
print("\n" + "="*70)
print("STEP 50A.1 — LABOUR COLUMNS")
print("="*70)
print(labour_clean.columns.tolist())


STEP 50A.1 — LABOUR COLUMNS
['district', 'year', 'labour_force_000', 'employed_000', 'unemployed_000', 'outside_lf_000', 'participation_rate_pct', 'unemployment_rate_pct', 'employment_pop_ratio_pct', 'original_district', 'canonical_district', 'district_id']


In [136]:
# check data types
print("\n" + "="*70)
print("STEP 50A.2 — LABOUR DATA TYPES")
print("="*70)
print(labour_clean.dtypes)


STEP 50A.2 — LABOUR DATA TYPES
district                        str
year                          Int64
labour_force_000            float64
employed_000                float64
unemployed_000              float64
outside_lf_000              float64
participation_rate_pct      float64
unemployment_rate_pct       float64
employment_pop_ratio_pct    float64
original_district               str
canonical_district              str
district_id                   int64
dtype: object


In [137]:
# check for missing values
print("\n" + "="*70)
print("STEP 50A.3 — LABOUR MISSING VALUES")
print("="*70)
print(
    labour_clean.isna().sum()
)


STEP 50A.3 — LABOUR MISSING VALUES
district                    0
year                        0
labour_force_000            0
employed_000                0
unemployed_000              0
outside_lf_000              0
participation_rate_pct      0
unemployment_rate_pct       0
employment_pop_ratio_pct    0
original_district           0
canonical_district          0
district_id                 0
dtype: int64


In [138]:
# check whether numeric columns cntain text
print("\n" + "="*70)
print("STEP 50A.4 — CHECK LABOUR VALUES")
print("="*70)
numeric_candidate_columns = [
    "labour_force_000",
    "employed_000",
    "unemployed_000",
    "outside_lf_000",
    "participation_rate_pct",
    "unemployment_rate_pct",
    "employment_pop_ratio_pct"
]
for col in numeric_candidate_columns:
    print("\n" + "-"*60)
    print(col)

    print("Data type:", labour_clean[col].dtype)

    print("First 10 values:")
    print(
        labour_clean[col].head(10).tolist()
    )



STEP 50A.4 — CHECK LABOUR VALUES

------------------------------------------------------------
labour_force_000
Data type: float64
First 10 values:
[60.5, 60.3, 52.5, 51.5, 52.8, 55.7, 57.4, 42.5, 43.3, 35.2]

------------------------------------------------------------
employed_000
Data type: float64
First 10 values:
[58.8, 58.4, 50.7, 50.0, 51.8, 54.9, 56.5, 41.4, 42.1, 33.9]

------------------------------------------------------------
unemployed_000
Data type: float64
First 10 values:
[1.7, 1.9, 1.8, 1.4, 1.1, 0.8, 0.9, 1.1, 1.2, 1.3]

------------------------------------------------------------
outside_lf_000
Data type: float64
First 10 values:
[30.8, 31.6, 26.8, 27.5, 28.0, 28.6, 29.1, 27.0, 26.9, 29.2]

------------------------------------------------------------
participation_rate_pct
Data type: float64
First 10 values:
[66.3, 65.6, 66.2, 65.2, 65.4, 66.1, 66.3, 61.1, 61.7, 54.7]

------------------------------------------------------------
unemployment_rate_pct
Data type: flo

In [139]:
# check whether "-" exists
print("\n" + "="*70)
print("STEP 50A.5 — CHECK FOR DASH VALUES")
print("="*70)

for col in numeric_candidate_columns:

    dash_count = (
        labour_clean[col]
        .astype(str)
        .str.strip()
        .eq("-")
        .sum()
    )

    print(
        f"{col:30s} → '-' values: {dash_count}"
    )


STEP 50A.5 — CHECK FOR DASH VALUES
labour_force_000               → '-' values: 0
employed_000                   → '-' values: 0
unemployed_000                 → '-' values: 0
outside_lf_000                 → '-' values: 0
participation_rate_pct         → '-' values: 0
unemployment_rate_pct          → '-' values: 0
employment_pop_ratio_pct       → '-' values: 0


In [140]:
# check whether commas or % exists
print("\n" + "="*70)
print("STEP 50A.6 — CHECK SPECIAL CHARACTERS")
print("="*70)

for col in numeric_candidate_columns:

    values = (
        labour_clean[col]
        .astype(str)
    )

    comma_count = values.str.contains(
        ",",
        regex=False,
        na=False
    ).sum()

    percent_count = values.str.contains(
        "%",
        regex=False,
        na=False
    ).sum()

    print(
        f"{col:30s} → "
        f"commas: {comma_count}, "
        f"percent signs: {percent_count}"
    )


STEP 50A.6 — CHECK SPECIAL CHARACTERS
labour_force_000               → commas: 0, percent signs: 0
employed_000                   → commas: 0, percent signs: 0
unemployed_000                 → commas: 0, percent signs: 0
outside_lf_000                 → commas: 0, percent signs: 0
participation_rate_pct         → commas: 0, percent signs: 0
unemployment_rate_pct          → commas: 0, percent signs: 0
employment_pop_ratio_pct       → commas: 0, percent signs: 0


In [141]:
# check whether numeric ranges
print("\n" + "="*70)
print("STEP 50A.7 — LABOUR NUMERIC SUMMARY")
print("="*70)

print(
    labour_clean[numeric_candidate_columns].describe().T
)


STEP 50A.7 — LABOUR NUMERIC SUMMARY
                          count       mean        std   min   25%   50%   75%  \
labour_force_000           77.0  65.931169  64.620138  18.8  39.9  45.2  57.1   
employed_000               77.0  64.220779  63.031695  18.3  38.7  43.9  56.0   
unemployed_000             77.0   1.710390   1.661054   0.3   1.0   1.2   1.7   
outside_lf_000             77.0  34.745455  25.396879   7.2  25.1  27.7  31.4   
participation_rate_pct     77.0  63.854545   5.023276  52.7  61.9  64.4  66.2   
unemployment_rate_pct      77.0   2.614286   0.651257   1.2   2.2   2.7   2.9   
employment_pop_ratio_pct   77.0  62.194805   4.957554  50.7  60.0  62.7  64.6   

                            max  
labour_force_000          285.0  
employed_000              279.7  
unemployed_000              8.5  
outside_lf_000            115.7  
participation_rate_pct     73.0  
unemployment_rate_pct       4.0  
employment_pop_ratio_pct   70.9  


In [142]:
# check the mathematical relationship
print("\n" + "="*70)
print("STEP 50A.8 — LABOUR FORCE CONSISTENCY CHECK")
print("="*70)

labour_clean["calculated_labour_force_000"] = (
    labour_clean["employed_000"]
    +
    labour_clean["unemployed_000"]
)

labour_clean["labour_force_difference"] = (
    labour_clean["labour_force_000"]
    -
    labour_clean["calculated_labour_force_000"]
)

print(
    labour_clean[
        [
            "district",
            "year",
            "labour_force_000",
            "employed_000",
            "unemployed_000",
            "calculated_labour_force_000",
            "labour_force_difference"
        ]
    ].head(20)
)
# checking for large discrepancies.
print("\nLargest absolute difference:")
print(
    labour_clean["labour_force_difference"]
    .abs()
    .max()
)


STEP 50A.8 — LABOUR FORCE CONSISTENCY CHECK
             district  year  labour_force_000  employed_000  unemployed_000  \
0             Bentong  2018              60.5          58.8             1.7   
1             Bentong  2019              60.3          58.4             1.9   
2             Bentong  2020              52.5          50.7             1.8   
3             Bentong  2021              51.5          50.0             1.4   
4             Bentong  2022              52.8          51.8             1.1   
5             Bentong  2023              55.7          54.9             0.8   
6             Bentong  2024              57.4          56.5             0.9   
7                Bera  2018              42.5          41.4             1.1   
8                Bera  2019              43.3          42.1             1.2   
9                Bera  2020              35.2          33.9             1.3   
10               Bera  2021              34.2          32.9             1.3   
11     

In [143]:
# check percentages ranges - to make sure the values makes sense
print("\n" + "="*70)
print("STEP 50A.9 — RATE RANGE CHECK")
print("="*70)

rate_columns = [
    "participation_rate_pct",
    "unemployment_rate_pct",
    "employment_pop_ratio_pct"
]

for col in rate_columns:

    print(f"\n{col}")

    print("Minimum:", labour_clean[col].min())
    print("Maximum:", labour_clean[col].max())


STEP 50A.9 — RATE RANGE CHECK

participation_rate_pct
Minimum: 52.7
Maximum: 73.0

unemployment_rate_pct
Minimum: 1.2
Maximum: 4.0

employment_pop_ratio_pct
Minimum: 50.7
Maximum: 70.9


In [144]:
# remove temporary validation
labour_clean = labour_clean.drop(
    columns=[
        "calculated_labour_force_000",
        "labour_force_difference"
    ]
)
print("\nTemporary validation columns removed.")



Temporary validation columns removed.


In [145]:
print(labour_clean.columns.tolist())

['district', 'year', 'labour_force_000', 'employed_000', 'unemployed_000', 'outside_lf_000', 'participation_rate_pct', 'unemployment_rate_pct', 'employment_pop_ratio_pct', 'original_district', 'canonical_district', 'district_id']


In [146]:
# confirm LABOUR is clean
print("\n" + "="*70)
print("STEP 50A.11 — LABOUR CLEANING CHECK")
print("="*70)

print("Shape:", labour_clean.shape)

print("\nData types:")
print(labour_clean.dtypes)

print("\nMissing values:")
print(labour_clean.isna().sum())


STEP 50A.11 — LABOUR CLEANING CHECK
Shape: (77, 12)

Data types:
district                        str
year                          Int64
labour_force_000            float64
employed_000                float64
unemployed_000              float64
outside_lf_000              float64
participation_rate_pct      float64
unemployment_rate_pct       float64
employment_pop_ratio_pct    float64
original_district               str
canonical_district              str
district_id                   int64
dtype: object

Missing values:
district                    0
year                        0
labour_force_000            0
employed_000                0
unemployed_000              0
outside_lf_000              0
participation_rate_pct      0
unemployment_rate_pct       0
employment_pop_ratio_pct    0
original_district           0
canonical_district          0
district_id                 0
dtype: int64


INCOME DATASET

In [147]:
print("\n" + "="*70)
print("STEP 50B — INCOME NUMERIC INSPECTION")
print("="*70)

print("\nColumns:")
print(income_clean.columns.tolist())

print("\nData types:")
print(income_clean.dtypes)

print("\nMissing values:")
print(income_clean.isna().sum())

print("\nFirst 10 rows:")
print(income_clean.head(10))


STEP 50B — INCOME NUMERIC INSPECTION

Columns:
['district', 'year', 'income_mean_rm', 'income_median_rm', 'original_district', 'canonical_district', 'district_id']

Data types:
district                str
year                  Int64
income_mean_rm        int64
income_median_rm      int64
original_district       str
canonical_district      str
district_id           int64
dtype: object

Missing values:
district              0
year                  0
income_mean_rm        0
income_median_rm      0
original_district     0
canonical_district    0
district_id           0
dtype: int64

First 10 rows:
            district  year  income_mean_rm  income_median_rm  \
0            Bentong  2019            5300              4220   
1            Bentong  2022            5563              4691   
2            Bentong  2024            6278              5015   
3               Bera  2019            4566              3636   
4               Bera  2022            4567              3866   
5             

In [148]:
# check for dashes
print("\n" + "="*70)
print("INCOME DASH CHECK")
print("="*70)

for col in income_clean.columns:

    dash_count = (
        income_clean[col]
        .astype(str)
        .str.strip()
        .eq("-")
        .sum()
    )

    if dash_count > 0:
        print(
            f"{col:30s} → '-' values: {dash_count}"
        )


INCOME DASH CHECK


In [149]:
print("\n" + "="*70)
print("INCOME NUMERIC SUMMARY")
print("="*70)

print(
    income_clean.describe(include="all").T
)


INCOME NUMERIC SUMMARY
                   count unique      top  freq         mean         std  \
district              33     11  Bentong     3          NaN         NaN   
year                33.0   <NA>     <NA>  <NA>  2021.666667    2.086664   
income_mean_rm      33.0    NaN      NaN   NaN  5398.393939  785.409803   
income_median_rm    33.0    NaN      NaN   NaN  4444.272727  662.942271   
original_district     33     11  Bentong     3          NaN         NaN   
canonical_district    33     11  Bentong     3          NaN         NaN   
district_id         33.0    NaN      NaN   NaN        606.0    3.211308   

                       min     25%     50%     75%     max  
district               NaN     NaN     NaN     NaN     NaN  
year                2019.0  2019.0  2022.0  2024.0  2024.0  
income_mean_rm      4452.0  4959.0  5180.0  5563.0  7071.0  
income_median_rm    3579.0  4053.0  4283.0  4691.0  5926.0  
original_district      NaN     NaN     NaN     NaN     NaN  
canonical

In [150]:
print("\n" + "=" * 70)
print("INCOME MISSING VALUE CHECK")
print("=" * 70)

income_numeric_cols = [
    "year",
    "income_mean_rm",
    "income_median_rm"
]

print("\nMissing values by numeric variable:")
print(income[income_numeric_cols].isna().sum())

print("\nMissing values by percentage:")
print(
    (income[income_numeric_cols].isna().mean() * 100)
    .round(2)
)


INCOME MISSING VALUE CHECK

Missing values by numeric variable:
year                0
income_mean_rm      0
income_median_rm    0
dtype: int64

Missing values by percentage:
year                0.0
income_mean_rm      0.0
income_median_rm    0.0
dtype: float64


In [151]:
# income logical consistency
# Mean household income ≥ Median household income
print("\n" + "=" * 70)
print("INCOME LOGICAL CONSISTENCY CHECK")
print("=" * 70)

# Check whether mean income is greater than or equal to median income
income_mean_below_median = income[
    income["income_mean_rm"] < income["income_median_rm"]
]

print("\nRecords where mean income < median income:")
print(len(income_mean_below_median))

if len(income_mean_below_median) > 0:
    print("\nPotential inconsistencies:")
    print(
        income_mean_below_median[
            ["district", "year",
             "income_mean_rm", "income_median_rm"]
        ]
    )
else:
    print("PASS: Mean income is >= median income for all records.")


INCOME LOGICAL CONSISTENCY CHECK

Records where mean income < median income:
0
PASS: Mean income is >= median income for all records.


In [152]:
# income duplicate check
print("\n" + "=" * 70)
print("INCOME DUPLICATE CHECK")
print("=" * 70)

duplicates = income[
    income.duplicated(
        subset=["district", "year"],
        keep=False
    )
]

print(f"\nDuplicate district-year records: {len(duplicates)}")

if len(duplicates) > 0:
    print("\nDUPLICATES FOUND:")
    print(duplicates.sort_values(["district", "year"]))
else:
    print("PASS: No duplicate district-year records.")


INCOME DUPLICATE CHECK

Duplicate district-year records: 0
PASS: No duplicate district-year records.


In [153]:
# year and district coverage
print("\n" + "=" * 70)
print("INCOME YEAR & DISTRICT COVERAGE")
print("=" * 70)

print("\nYears:")
print(sorted(income["year"].unique()))

print("\nNumber of districts:")
print(income["district"].nunique())

print("\nRecords per year:")
print(income["year"].value_counts().sort_index())

print("\nRecords per district:")
print(income["district"].value_counts().sort_index())


INCOME YEAR & DISTRICT COVERAGE

Years:
[np.int64(2019), np.int64(2022), np.int64(2024)]

Number of districts:
11

Records per year:
year
2019    11
2022    11
2024    11
Name: count, dtype: int64

Records per district:
district
Bentong              3
Bera                 3
Cameron Highlands    3
Jerantut             3
Kuantan              3
Lipis                3
Maran                3
Pekan                3
Raub                 3
Rompin               3
Temerloh             3
Name: count, dtype: int64


In [154]:
# income range check
print("\n" + "=" * 70)
print("INCOME RANGE CHECK")
print("=" * 70)

for col in ["income_mean_rm", "income_median_rm"]:
    print(f"\n{col}:")
    print(f"Minimum: RM {income[col].min():,.2f}")
    print(f"Maximum: RM {income[col].max():,.2f}")
    print(f"Negative values: {(income[col] < 0).sum()}")
    print(f"Zero values: {(income[col] == 0).sum()}")


INCOME RANGE CHECK

income_mean_rm:
Minimum: RM 4,452.00
Maximum: RM 7,071.00
Negative values: 0
Zero values: 0

income_median_rm:
Minimum: RM 3,579.00
Maximum: RM 5,926.00
Negative values: 0
Zero values: 0


POVERTY

In [155]:
print("\n" + "=" * 70)
print("POVERTY NUMERIC SUMMARY")
print("=" * 70)

print(poverty.describe(include="all").T)


POVERTY NUMERIC SUMMARY
                     count unique      top freq         mean       std  \
district                33     11  Bentong    3          NaN       NaN   
year                  33.0    NaN      NaN  NaN  2021.666667  2.086664   
poverty_absolute_pct  33.0    NaN      NaN  NaN     6.221212  3.132666   
poverty_relative_pct  33.0    NaN      NaN  NaN     7.236364  5.194337   

                         min     25%     50%     75%     max  
district                 NaN     NaN     NaN     NaN     NaN  
year                  2019.0  2019.0  2022.0  2024.0  2024.0  
poverty_absolute_pct     0.4     3.4     6.4     8.8    12.0  
poverty_relative_pct     0.0     3.0     8.4    10.0    18.5  


In [156]:
# check missing values, duplicates, and range check
print("\n" + "=" * 70)
print("POVERTY VALIDATION CHECKS")
print("=" * 70)

# --- Missing values ---
poverty_cols = [
    "year",
    "poverty_absolute_pct",
    "poverty_relative_pct"
]

print("\n[50C.2] Missing values:")
print(poverty[poverty_cols].isna().sum())

# --- Duplicate district-year ---
duplicates = poverty[
    poverty.duplicated(
        subset=["district", "year"],
        keep=False
    )
]

print("\n[50C.3] Duplicate district-year records:")
print(len(duplicates))

# --- Range check ---
print("\n[50C.4] Range check:")

for col in ["poverty_absolute_pct", "poverty_relative_pct"]:
    print(f"\n{col}")
    print(f"Minimum: {poverty[col].min():.2f}%")
    print(f"Maximum: {poverty[col].max():.2f}%")
    print(f"Below 0%: {(poverty[col] < 0).sum()}")
    print(f"Above 100%: {(poverty[col] > 100).sum()}")


POVERTY VALIDATION CHECKS

[50C.2] Missing values:
year                    0
poverty_absolute_pct    0
poverty_relative_pct    0
dtype: int64

[50C.3] Duplicate district-year records:
0

[50C.4] Range check:

poverty_absolute_pct
Minimum: 0.40%
Maximum: 12.00%
Below 0%: 0
Above 100%: 0

poverty_relative_pct
Minimum: 0.00%
Maximum: 18.50%
Below 0%: 0
Above 100%: 0


In [157]:
# full validation
# ======================================================================
# STEP 50C — POVERTY FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50C — POVERTY FULL VALIDATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 50C.2 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50C.2] MISSING VALUES")

poverty_numeric = [
    "year",
    "poverty_absolute_pct",
    "poverty_relative_pct"
]

print(poverty[poverty_numeric].isna().sum())


# ----------------------------------------------------------------------
# 50C.3 — DUPLICATE DISTRICT-YEAR
# ----------------------------------------------------------------------

print("\n[50C.3] DUPLICATE DISTRICT-YEAR CHECK")

duplicates = poverty[
    poverty.duplicated(
        subset=["district", "year"],
        keep=False
    )
]

print(f"Duplicate records: {len(duplicates)}")


# ----------------------------------------------------------------------
# 50C.4 — YEAR COVERAGE
# ----------------------------------------------------------------------

print("\n[50C.4] YEAR COVERAGE")

print("Years:", sorted(poverty["year"].unique()))
print("Records per year:")
print(poverty["year"].value_counts().sort_index())


# ----------------------------------------------------------------------
# 50C.5 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50C.5] DISTRICT COVERAGE")

print("Number of districts:", poverty["district"].nunique())
print("Records per district:")
print(poverty["district"].value_counts().sort_index())


# ----------------------------------------------------------------------
# 50C.6 — RANGE CHECK
# ----------------------------------------------------------------------

print("\n[50C.6] RANGE CHECK")

for col in [
    "poverty_absolute_pct",
    "poverty_relative_pct"
]:
    print(f"\n{col}")
    print(f"Minimum: {poverty[col].min():.2f}%")
    print(f"Maximum: {poverty[col].max():.2f}%")
    print(f"Below 0%: {(poverty[col] < 0).sum()}")
    print(f"Above 100%: {(poverty[col] > 100).sum()}")


# ----------------------------------------------------------------------
# 50C.7 — LOGICAL CONSISTENCY
# ----------------------------------------------------------------------

print("\n[50C.7] LOGICAL CONSISTENCY")

# Absolute and relative poverty should both lie between 0 and 100.
invalid_absolute = poverty[
    (poverty["poverty_absolute_pct"] < 0) |
    (poverty["poverty_absolute_pct"] > 100)
]

invalid_relative = poverty[
    (poverty["poverty_relative_pct"] < 0) |
    (poverty["poverty_relative_pct"] > 100)
]

print("Invalid absolute poverty records:", len(invalid_absolute))
print("Invalid relative poverty records:", len(invalid_relative))


# ----------------------------------------------------------------------
# 50C.8 — DATA TYPE CHECK
# ----------------------------------------------------------------------

print("\n[50C.8] DATA TYPES")

print(poverty.dtypes)


# ----------------------------------------------------------------------
# 50C.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("POVERTY VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(poverty))
print("Districts:", poverty["district"].nunique())
print("Years:", sorted(poverty["year"].unique()))


STEP 50C — POVERTY FULL VALIDATION

[50C.2] MISSING VALUES
year                    0
poverty_absolute_pct    0
poverty_relative_pct    0
dtype: int64

[50C.3] DUPLICATE DISTRICT-YEAR CHECK
Duplicate records: 0

[50C.4] YEAR COVERAGE
Years: [np.int64(2019), np.int64(2022), np.int64(2024)]
Records per year:
year
2019    11
2022    11
2024    11
Name: count, dtype: int64

[50C.5] DISTRICT COVERAGE
Number of districts: 11
Records per district:
district
Bentong              3
Bera                 3
Cameron Highlands    3
Jerantut             3
Kuantan              3
Lipis                3
Maran                3
Pekan                3
Raub                 3
Rompin               3
Temerloh             3
Name: count, dtype: int64

[50C.6] RANGE CHECK

poverty_absolute_pct
Minimum: 0.40%
Maximum: 12.00%
Below 0%: 0
Above 100%: 0

poverty_relative_pct
Minimum: 0.00%
Maximum: 18.50%
Below 0%: 0
Above 100%: 0

[50C.7] LOGICAL CONSISTENCY
Invalid absolute poverty records: 0
Invalid relative povert

POPULATION

In [158]:
print("\n" + "=" * 70)
print("STEP 50D — POPULATION FULL VALIDATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 50D.1 — NUMERIC / STRUCTURAL SUMMARY
# ----------------------------------------------------------------------

print("\n[50D.1] DATA SUMMARY")
print(population.describe(include="all").T)


# ----------------------------------------------------------------------
# 50D.2 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50D.2] MISSING VALUES")

print(population.isna().sum())


# ----------------------------------------------------------------------
# 50D.3 — DUPLICATE CHECK
# ----------------------------------------------------------------------

print("\n[50D.3] DUPLICATE CHECK")

print("Exact duplicate rows:", population.duplicated().sum())

# Check duplicates at the expected identifying level
population_key = [
    "district",
    "date",
    "sex",
    "age",
    "ethnicity"
]

duplicates = population[
    population.duplicated(
        subset=population_key,
        keep=False
    )
]

print("Duplicate district-date-sex-age-ethnicity records:",
      len(duplicates))


# ----------------------------------------------------------------------
# 50D.4 — DATE COVERAGE
# ----------------------------------------------------------------------

print("\n[50D.4] DATE COVERAGE")

print("Date data type:")
print(population["date"].dtype)

print("\nUnique dates:")
print(sorted(population["date"].unique()))


# ----------------------------------------------------------------------
# 50D.5 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50D.5] DISTRICT COVERAGE")

print("Number of districts:",
      population["district"].nunique())

print("\nDistricts:")
print(sorted(population["district"].dropna().unique()))


# ----------------------------------------------------------------------
# 50D.6 — CATEGORICAL COVERAGE
# ----------------------------------------------------------------------

print("\n[50D.6] CATEGORY COVERAGE")

for col in ["sex", "age", "ethnicity"]:
    print(f"\n{col}:")
    print("Number of categories:",
          population[col].nunique(dropna=True))
    print(population[col].value_counts(dropna=False).head(20))


# ----------------------------------------------------------------------
# 50D.7 — POPULATION RANGE CHECK
# ----------------------------------------------------------------------

print("\n[50D.7] POPULATION RANGE CHECK")

print(f"Minimum population_000: {population['population_000'].min()}")
print(f"Maximum population_000: {population['population_000'].max()}")

print("Negative values:",
      (population["population_000"] < 0).sum())

print("Zero values:",
      (population["population_000"] == 0).sum())


# ----------------------------------------------------------------------
# 50D.8 — DATA TYPES
# ----------------------------------------------------------------------

print("\n[50D.8] DATA TYPES")

print(population.dtypes)


# ----------------------------------------------------------------------
# 50D.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("POPULATION VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(population))
print("Districts:", population["district"].nunique())
print("Dates:", population["date"].nunique())
print("Sex categories:", population["sex"].nunique(dropna=True))
print("Age categories:", population["age"].nunique(dropna=True))
print("Ethnicity categories:", population["ethnicity"].nunique(dropna=True))


STEP 50D — POPULATION FULL VALIDATION

[50D.1] DATA SUMMARY
                  count unique         top   freq      mean        std  min  \
state             26334      1      Pahang  26334       NaN        NaN  NaN   
district          26334     12     Bentong   2394       NaN        NaN  NaN   
date              26334      6  2020-01-01   4389       NaN        NaN  NaN   
sex               26334      3        both   8778       NaN        NaN  NaN   
age               26334     19     overall   1386       NaN        NaN  NaN   
ethnicity         26334      7     overall   3762       NaN        NaN  NaN   
population_000  26334.0    NaN         NaN    NaN  2.973825  15.902924  0.0   

                25%  50%  75%    max  
state           NaN  NaN  NaN    NaN  
district        NaN  NaN  NaN    NaN  
date            NaN  NaN  NaN    NaN  
sex             NaN  NaN  NaN    NaN  
age             NaN  NaN  NaN    NaN  
ethnicity       NaN  NaN  NaN    NaN  
population_000  0.0  0.3  1.7  57

In [159]:
print("\n" + "=" * 70)
print("POPULATION KEY VALIDATION RESULTS")
print("=" * 70)

print("\n[1] Missing values")
print(population.isna().sum())

print("\n[2] Exact duplicate rows")
print(population.duplicated().sum())

population_key = [
    "district",
    "date",
    "sex",
    "age",
    "ethnicity"
]

print("\n[3] Duplicate identifying records")
print(
    population.duplicated(
        subset=population_key,
        keep=False
    ).sum()
)

print("\n[4] Dates")
print(sorted(population["date"].unique()))

print("\n[5] Districts")
print(sorted(population["district"].unique()))

print("\n[6] Negative population values")
print((population["population_000"] < 0).sum())

print("\n[7] Zero population values")
print((population["population_000"] == 0).sum())

print("\n[8] Data types")
print(population.dtypes)

print("\n" + "=" * 70)
print("POPULATION CHECK COMPLETE")
print("=" * 70)


POPULATION KEY VALIDATION RESULTS

[1] Missing values
state             0
district          0
date              0
sex               0
age               0
ethnicity         0
population_000    0
dtype: int64

[2] Exact duplicate rows
0

[3] Duplicate identifying records
0

[4] Dates
['2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01']

[5] Districts
['Bentong', 'Bera', 'Cameron Highland', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

[6] Negative population values
0

[7] Zero population values
6999

[8] Data types
state                 str
district              str
date                  str
sex                   str
age                   str
ethnicity             str
population_000    float64
dtype: object

POPULATION CHECK COMPLETE


AMENITIES

In [160]:
# ======================================================================
# STEP 50E — AMENITIES FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50E — AMENITIES FULL VALIDATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 50E.1 — DATA STRUCTURE
# ----------------------------------------------------------------------

print("\n[50E.1] DATA SUMMARY")
print(amenities.describe(include="all").T)


# ----------------------------------------------------------------------
# 50E.2 — COLUMN NAMES
# ----------------------------------------------------------------------

print("\n[50E.2] COLUMNS")
print(amenities.columns.tolist())


# ----------------------------------------------------------------------
# 50E.3 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50E.3] MISSING VALUES")
print(amenities.isna().sum())


# ----------------------------------------------------------------------
# 50E.4 — EXACT DUPLICATES
# ----------------------------------------------------------------------

print("\n[50E.4] EXACT DUPLICATES")
print("Exact duplicate rows:", amenities.duplicated().sum())


# ----------------------------------------------------------------------
# 50E.5 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50E.5] DISTRICT COVERAGE")
print("Number of districts:", amenities["district"].nunique())
print("Districts:")
print(sorted(amenities["district"].unique()))


# ----------------------------------------------------------------------
# 50E.6 — YEAR / DATE COVERAGE
# ----------------------------------------------------------------------

print("\n[50E.6] YEAR / DATE COVERAGE")

if "year" in amenities.columns:
    print("Years:", sorted(amenities["year"].unique()))
    print("\nRecords per year:")
    print(amenities["year"].value_counts().sort_index())

elif "date" in amenities.columns:
    print("Dates:", sorted(amenities["date"].unique()))
    print("\nRecords per date:")
    print(amenities["date"].value_counts().sort_index())


# ----------------------------------------------------------------------
# 50E.7 — NUMERIC RANGE CHECK
# ----------------------------------------------------------------------

print("\n[50E.7] NUMERIC RANGE CHECK")

numeric_cols = amenities.select_dtypes(
    include="number"
).columns.tolist()

print("Numeric columns:", numeric_cols)

for col in numeric_cols:
    print(
        f"{col}: "
        f"min={amenities[col].min()}, "
        f"max={amenities[col].max()}, "
        f"negative={(amenities[col] < 0).sum()}"
    )


# ----------------------------------------------------------------------
# 50E.8 — DATA TYPES
# ----------------------------------------------------------------------

print("\n[50E.8] DATA TYPES")
print(amenities.dtypes)


# ----------------------------------------------------------------------
# 50E.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("AMENITIES VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(amenities))
print("Districts:", amenities["district"].nunique())

if "year" in amenities.columns:
    print("Years:", sorted(amenities["year"].unique()))
elif "date" in amenities.columns:
    print("Dates:", amenities["date"].nunique())


STEP 50E — AMENITIES FULL VALIDATION

[50E.1] DATA SUMMARY
              count unique            top freq       mean       std   min  \
date             48      4     2016-01-01   12        NaN       NaN   NaN   
state            48      1         Pahang   48        NaN       NaN   NaN   
district         48     12  All Districts    4        NaN       NaN   NaN   
piped_water_%  47.0    NaN            NaN  NaN  97.262766  3.403094  84.3   
sanitation_%   47.0    NaN            NaN  NaN  99.611277   1.42494  91.1   
electricity_%  48.0    NaN            NaN  NaN  99.808542  0.984676  93.5   

                 25%    50%    75%    max  
date             NaN    NaN    NaN    NaN  
state            NaN    NaN    NaN    NaN  
district         NaN    NaN    NaN    NaN  
piped_water_%   96.2   98.7  99.95  100.0  
sanitation_%   100.0  100.0  100.0  100.0  
electricity_%  100.0  100.0  100.0  100.0  

[50E.2] COLUMNS
['date', 'state', 'district', 'piped_water_%', 'sanitation_%', 'electricity

In [161]:
# ======================================================================
# STEP 50E.10 — AMENITIES ISSUE DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("AMENITIES ISSUE DIAGNOSTIC")
print("=" * 70)

# Missing values
print("\n[1] Missing-value records:")
print(
    amenities[
        amenities[
            ["piped_water_%", "sanitation_%", "electricity_%"]
        ].isna().any(axis=1)
    ]
)

# All Districts rows
print("\n[2] 'All Districts' records:")
print(
    amenities[
        amenities["district"].eq("All Districts")
    ]
)

# District list
print("\n[3] District list:")
print(sorted(amenities["district"].unique()))

# Date list
print("\n[4] Dates:")
print(sorted(amenities["date"].unique()))

# Range check
print("\n[5] Percentage range:")
for col in [
    "piped_water_%",
    "sanitation_%",
    "electricity_%"
]:
    print(
        f"{col}: "
        f"min={amenities[col].min()}, "
        f"max={amenities[col].max()}, "
        f"below 0={(amenities[col] < 0).sum()}, "
        f"above 100={(amenities[col] > 100).sum()}"
    )

print("\n" + "=" * 70)


AMENITIES ISSUE DIAGNOSTIC

[1] Missing-value records:
         date   state       district  piped_water_%  sanitation_%  \
3  2024-01-01  Pahang  All Districts            NaN           NaN   

   electricity_%  
3          100.0  

[2] 'All Districts' records:
         date   state       district  piped_water_%  sanitation_%  \
0  2016-01-01  Pahang  All Districts           97.8         99.75   
1  2019-01-01  Pahang  All Districts           97.9         99.98   
2  2022-01-01  Pahang  All Districts           98.3         99.40   
3  2024-01-01  Pahang  All Districts            NaN           NaN   

   electricity_%  
0         100.00  
1         100.00  
2          99.51  
3         100.00  

[3] District list:
['All Districts', 'Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

[4] Dates:
['2016-01-01', '2019-01-01', '2022-01-01', '2024-01-01']

[5] Percentage range:
piped_water_%: min=84.3, max=100.0, below 0=0,

In [162]:
# ==============================================================
# STEP 50E.11 — REMOVE AGGREGATE "ALL DISTRICTS" ROW
# ==============================================================

print("\n" + "=" * 70)
print("REMOVING AGGREGATE AMENITIES ROWS")
print("=" * 70)

before = len(amenities)

amenities = amenities[
    amenities["district"].ne("All Districts")
].copy()

after = len(amenities)

print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Rows removed: {before - after}")

print("\nRemaining districts:")
print(sorted(amenities["district"].unique()))


REMOVING AGGREGATE AMENITIES ROWS
Rows before: 48
Rows after:  44
Rows removed: 4

Remaining districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']


In [163]:
# ==============================================================
# STEP 50E.12 — AMENITIES FINAL CHECK
# ==============================================================

print("\n" + "=" * 70)
print("AMENITIES FINAL CHECK")
print("=" * 70)

print("\nMissing values:")
print(
    amenities[
        ["piped_water_%", "sanitation_%", "electricity_%"]
    ].isna().sum()
)

print("\nDuplicate district-date records:")
print(
    amenities.duplicated(
        subset=["district", "date"],
        keep=False
    ).sum()
)

print("\nRows:", len(amenities))
print("Districts:", amenities["district"].nunique())
print("Dates:", sorted(amenities["date"].unique()))


AMENITIES FINAL CHECK

Missing values:
piped_water_%    0
sanitation_%     0
electricity_%    0
dtype: int64

Duplicate district-date records:
0

Rows: 44
Districts: 11
Dates: ['2016-01-01', '2019-01-01', '2022-01-01', '2024-01-01']


ENROLMENT

In [164]:
print(enrollment.columns.tolist())

['state', 'district', 'stage', 'sex', 'date', 'students']


In [165]:
# ======================================================================
# STEP 50F — ENROLLMENT FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50F — ENROLLMENT FULL VALIDATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 50F.1 — DATA SUMMARY
# ----------------------------------------------------------------------

print("\n[50F.1] DATA SUMMARY")
print(enrollment.describe(include="all").T)


# ----------------------------------------------------------------------
# 50F.2 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50F.2] MISSING VALUES")
print(enrollment.isna().sum())


# ----------------------------------------------------------------------
# 50F.3 — EXACT DUPLICATES
# ----------------------------------------------------------------------

print("\n[50F.3] EXACT DUPLICATES")
print("Exact duplicate rows:", enrollment.duplicated().sum())


# ----------------------------------------------------------------------
# 50F.4 — DUPLICATE IDENTIFYING RECORDS
# ----------------------------------------------------------------------

print("\n[50F.4] DUPLICATE CHECK")

enrollment_key = [
    "district",
    "stage",
    "sex",
    "date"
]

duplicates = enrollment[
    enrollment.duplicated(
        subset=enrollment_key,
        keep=False
    )
]

print(
    "Duplicate district-stage-sex-date records:",
    len(duplicates)
)


# ----------------------------------------------------------------------
# 50F.5 — DATE COVERAGE
# ----------------------------------------------------------------------

print("\n[50F.5] DATE COVERAGE")

print("Dates:")
print(sorted(enrollment["date"].unique()))

print("\nRecords per date:")
print(enrollment["date"].value_counts().sort_index())


# ----------------------------------------------------------------------
# 50F.6 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50F.6] DISTRICT COVERAGE")

print("Number of districts:",
      enrollment["district"].nunique())

print("\nDistricts:")
print(sorted(enrollment["district"].unique()))


# ----------------------------------------------------------------------
# 50F.7 — STAGE AND SEX COVERAGE
# ----------------------------------------------------------------------

print("\n[50F.7] CATEGORY COVERAGE")

print("\nStage:")
print(enrollment["stage"].value_counts(dropna=False))

print("\nSex:")
print(enrollment["sex"].value_counts(dropna=False))


# ----------------------------------------------------------------------
# 50F.8 — STUDENT COUNT RANGE
# ----------------------------------------------------------------------

print("\n[50F.8] STUDENT COUNT RANGE")

print("Minimum students:", enrollment["students"].min())
print("Maximum students:", enrollment["students"].max())

print(
    "Negative values:",
    (enrollment["students"] < 0).sum()
)

print(
    "Zero values:",
    (enrollment["students"] == 0).sum()
)


# ----------------------------------------------------------------------
# 50F.9 — DATA TYPES
# ----------------------------------------------------------------------

print("\n[50F.9] DATA TYPES")
print(enrollment.dtypes)


# ----------------------------------------------------------------------
# 50F.10 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("ENROLLMENT VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(enrollment))
print("Districts:", enrollment["district"].nunique())
print("Dates:", enrollment["date"].nunique())
print("Stages:", enrollment["stage"].nunique())
print("Sex categories:", enrollment["sex"].nunique())


STEP 50F — ENROLLMENT FULL VALIDATION

[50F.1] DATA SUMMARY
          count unique            top freq         mean           std   min  \
state       936      1         Pahang  936          NaN           NaN   NaN   
district    936     13  All Districts   78          NaN           NaN   NaN   
stage       936      3        primary  324          NaN           NaN   NaN   
sex         936      3           both  312          NaN           NaN   NaN   
date        936      9     2018-01-01  108          NaN           NaN   NaN   
students  936.0    NaN            NaN  NaN  9839.303419  21298.052539  10.0   

             25%     50%     75%       max  
state        NaN     NaN     NaN       NaN  
district     NaN     NaN     NaN       NaN  
stage        NaN     NaN     NaN       NaN  
sex          NaN     NaN     NaN       NaN  
date         NaN     NaN     NaN       NaN  
students  571.75  4011.0  7369.5  152824.0  

[50F.2] MISSING VALUES
state       0
district    0
stage       0
sex 

In [166]:
# ======================================================================
# STEP 50F.11 — ENROLLMENT COVERAGE DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("ENROLLMENT COVERAGE DIAGNOSTIC")
print("=" * 70)

print("\n[1] Districts:")
print(sorted(enrollment["district"].unique()))

print("\n[2] Records per district:")
print(enrollment["district"].value_counts().sort_index())

print("\n[3] Dates:")
print(sorted(enrollment["date"].unique()))

print("\n[4] Duplicate district-stage-sex-date records:")
print(
    enrollment.duplicated(
        subset=["district", "stage", "sex", "date"],
        keep=False
    ).sum()
)

print("\n[5] Negative students:")
print((enrollment["students"] < 0).sum())

print("\n[6] Zero students:")
print((enrollment["students"] == 0).sum())

print("\n" + "=" * 70)


ENROLLMENT COVERAGE DIAGNOSTIC

[1] Districts:
['All Districts', 'Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuala Lipis', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

[2] Records per district:
district
All Districts        78
Bentong              78
Bera                 78
Cameron Highlands    78
Jerantut             78
Kuala Lipis          45
Kuantan              78
Lipis                33
Maran                78
Pekan                78
Raub                 78
Rompin               78
Temerloh             78
Name: count, dtype: int64

[3] Dates:
['2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-06-30', '2024-06-30', '2025-06-30']

[4] Duplicate district-stage-sex-date records:
0

[5] Negative students:
0

[6] Zero students:
0



In [167]:
# ======================================================================
# STEP 50F.12 — CLEAN ENROLLMENT DISTRICT NAMES
# ======================================================================

print("\n" + "=" * 70)
print("CLEANING ENROLLMENT DISTRICT NAMES")
print("=" * 70)

before = len(enrollment)

# Remove aggregate
enrollment = enrollment[
    enrollment["district"] != "All Districts"
].copy()

# Normalize Lipis to Kuala Lipis
enrollment["district"] = enrollment["district"].replace({
    "Lipis": "Kuala Lipis"
})

print("Rows before:", before)
print("Rows after:", len(enrollment))

print("\nDistricts after normalization:")
print(sorted(enrollment["district"].unique()))

print("\nRecords per district:")
print(enrollment["district"].value_counts().sort_index())


CLEANING ENROLLMENT DISTRICT NAMES
Rows before: 936
Rows after: 858

Districts after normalization:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuala Lipis', 'Kuantan', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

Records per district:
district
Bentong              78
Bera                 78
Cameron Highlands    78
Jerantut             78
Kuala Lipis          78
Kuantan              78
Maran                78
Pekan                78
Raub                 78
Rompin               78
Temerloh             78
Name: count, dtype: int64


In [168]:
print(crop.columns.tolist())

NameError: name 'crop' is not defined

In [ ]:
print([x for x in globals() if "crop" in x.lower()])

['crop_2017_2022', 'crop_2023', 'crop_clean', 'crop_2023_clean', 'unmapped_crop_2023', 'crop_crosswalk', 'crop_2023_crosswalk']


In [ ]:
# ======================================================================
# STEP 50G — CROP FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50G — CROP FULL VALIDATION")
print("=" * 70)


# ======================================================================
# 50G.1 — CROP 2017–2022 STRUCTURE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.1] CROP 2017–2022 SUMMARY")
print("-" * 70)

print(crop_2017_2022.describe(include="all").T)

print("\nColumns:")
print(crop_2017_2022.columns.tolist())

print("\nMissing values:")
print(crop_2017_2022.isna().sum())

print("\nExact duplicates:")
print(crop_2017_2022.duplicated().sum())


# ======================================================================
# 50G.2 — CROP 2017–2022 COVERAGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.2] CROP 2017–2022 COVERAGE")
print("-" * 70)

print("Unique districts:")
print(crop_2017_2022["district"].nunique())

print("\nDistricts:")
print(sorted(crop_2017_2022["district"].unique()))

print("\nYears:")
print(sorted(crop_2017_2022["year"].unique()))


# ======================================================================
# 50G.3 — CROP 2017–2022 NUMERIC RANGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.3] CROP 2017–2022 NUMERIC RANGE")
print("-" * 70)

numeric_cols = crop_2017_2022.select_dtypes(
    include="number"
).columns.tolist()

print("Numeric columns:", numeric_cols)

for col in numeric_cols:
    print(
        f"{col}: "
        f"min={crop_2017_2022[col].min()}, "
        f"max={crop_2017_2022[col].max()}, "
        f"negative={(crop_2017_2022[col] < 0).sum()}"
    )


# ======================================================================
# 50G.4 — CROP 2023 STRUCTURE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.4] CROP 2023 SUMMARY")
print("-" * 70)

print(crop_2023.describe(include="all").T)

print("\nColumns:")
print(crop_2023.columns.tolist())

print("\nMissing values:")
print(crop_2023.isna().sum())

print("\nExact duplicates:")
print(crop_2023.duplicated().sum())


# ======================================================================
# 50G.5 — CROP 2023 COVERAGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.5] CROP 2023 COVERAGE")
print("-" * 70)

print("Unique districts:")
print(crop_2023["district"].nunique())

print("\nDistricts:")
print(sorted(crop_2023["district"].unique()))


# ======================================================================
# 50G.6 — CROP 2023 NUMERIC RANGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.6] CROP 2023 NUMERIC RANGE")
print("-" * 70)

numeric_cols_2023 = crop_2023.select_dtypes(
    include="number"
).columns.tolist()

print("Numeric columns:", numeric_cols_2023)

for col in numeric_cols_2023:
    print(
        f"{col}: "
        f"min={crop_2023[col].min()}, "
        f"max={crop_2023[col].max()}, "
        f"negative={(crop_2023[col] < 0).sum()}"
    )


# ======================================================================
# 50G.7 — FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("CROP VALIDATION COMPLETE")
print("=" * 70)

print(
    "Crop 2017–2022 rows:",
    len(crop_2017_2022)
)

print(
    "Crop 2023 rows:",
    len(crop_2023)
)


STEP 50G — CROP FULL VALIDATION

----------------------------------------------------------------------
[50G.1] CROP 2017–2022 SUMMARY
----------------------------------------------------------------------
                    count unique         top  freq        mean          std  \
date                 1024      1  2017-01-01  1024         NaN          NaN   
state                1024      1      Pahang  1024         NaN          NaN   
district             1024     11     Kuantan    94         NaN          NaN   
crop_type            1024      6   vegetable   319         NaN          NaN   
crop_species         1024     94     cassava    11         NaN          NaN   
production_tonnes  1024.0    NaN         NaN   NaN  592.190476  4823.508954   

                   min  25%     50%       75%       max  
date               NaN  NaN     NaN       NaN       NaN  
state              NaN  NaN     NaN       NaN       NaN  
district           NaN  NaN     NaN       NaN       NaN  
crop_ty

KeyError: 'year'

In [ ]:
print("CROP 2017–2022 COLUMNS:")
print(crop_2017_2022.columns.tolist())

print("\nCROP 2023 COLUMNS:")
print(crop_2023.columns.tolist())

CROP 2017–2022 COLUMNS:
['date', 'state', 'district', 'crop_type', 'crop_species', 'production_tonnes']

CROP 2023 COLUMNS:
['district', 'paddy_production_tonnes', 'oil_palm_production_tonnes', 'rubber_production_tonnes', 'fruits_production_tonnes', 'vegetable_production_tonnes', 'kenaf_production_tonnes', 'cocoa_production_tonnes', 'pepper_production_tonnes', 'pineapple_production_tonnes', 'others_production_tonnes']


In [ ]:
# ======================================================================
# STEP 50G — CROP FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50G — CROP FULL VALIDATION")
print("=" * 70)


# ======================================================================
# 50G.1 — CROP 2017–2022 SUMMARY
# ======================================================================

print("\n" + "-" * 70)
print("[50G.1] CROP 2017–2022 SUMMARY")
print("-" * 70)

print("Rows:", len(crop_2017_2022))
print("Columns:", crop_2017_2022.columns.tolist())

print("\nMissing values:")
print(crop_2017_2022.isna().sum())

print("\nExact duplicates:")
print(crop_2017_2022.duplicated().sum())


# ======================================================================
# 50G.2 — CROP 2017–2022 COVERAGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.2] CROP 2017–2022 COVERAGE")
print("-" * 70)

print("Districts:",
      crop_2017_2022["district"].nunique())

print("\nDistricts:")
print(sorted(crop_2017_2022["district"].unique()))

print("\nDates:")
print(sorted(crop_2017_2022["date"].unique()))

print("\nCrop types:")
print(crop_2017_2022["crop_type"].value_counts(dropna=False))


# ======================================================================
# 50G.3 — CROP PRODUCTION RANGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.3] CROP 2017–2022 PRODUCTION RANGE")
print("-" * 70)

print(
    "Minimum production:",
    crop_2017_2022["production_tonnes"].min()
)

print(
    "Maximum production:",
    crop_2017_2022["production_tonnes"].max()
)

print(
    "Negative production:",
    (crop_2017_2022["production_tonnes"] < 0).sum()
)


# ======================================================================
# 50G.4 — CROP 2023 SUMMARY
# ======================================================================

print("\n" + "-" * 70)
print("[50G.4] CROP 2023 SUMMARY")
print("-" * 70)

print("Rows:", len(crop_2023))
print("Columns:", crop_2023.columns.tolist())

print("\nMissing values:")
print(crop_2023.isna().sum())

print("\nExact duplicates:")
print(crop_2023.duplicated().sum())


# ======================================================================
# 50G.5 — CROP 2023 DISTRICT COVERAGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.5] CROP 2023 DISTRICT COVERAGE")
print("-" * 70)

print("Number of districts:",
      crop_2023["district"].nunique())

print("\nDistricts:")
print(sorted(crop_2023["district"].unique()))


# ======================================================================
# 50G.6 — CROP 2023 PRODUCTION RANGE
# ======================================================================

print("\n" + "-" * 70)
print("[50G.6] CROP 2023 PRODUCTION RANGE")
print("-" * 70)

crop_2023_numeric = crop_2023.select_dtypes(
    include="number"
).columns.tolist()

print("Production columns:", crop_2023_numeric)

for col in crop_2023_numeric:
    print(
        f"{col}: "
        f"min={crop_2023[col].min()}, "
        f"max={crop_2023[col].max()}, "
        f"negative={(crop_2023[col] < 0).sum()}"
    )


# ======================================================================
# 50G.7 — CROP 2023 DUPLICATE DISTRICTS
# ======================================================================

print("\n" + "-" * 70)
print("[50G.7] CROP 2023 DISTRICT DUPLICATES")
print("-" * 70)

print(
    "Duplicate district records:",
    crop_2023["district"].duplicated(keep=False).sum()
)


# ======================================================================
# 50G.8 — FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("CROP VALIDATION COMPLETE")
print("=" * 70)

print("2017–2022 rows:", len(crop_2017_2022))
print("2017–2022 districts:", crop_2017_2022["district"].nunique())
print("2023 rows:", len(crop_2023))
print("2023 districts:", crop_2023["district"].nunique())


STEP 50G — CROP FULL VALIDATION

----------------------------------------------------------------------
[50G.1] CROP 2017–2022 SUMMARY
----------------------------------------------------------------------
Rows: 1024
Columns: ['date', 'state', 'district', 'crop_type', 'crop_species', 'production_tonnes']

Missing values:
date                 0
state                0
district             0
crop_type            0
crop_species         0
production_tonnes    0
dtype: int64

Exact duplicates:
0

----------------------------------------------------------------------
[50G.2] CROP 2017–2022 COVERAGE
----------------------------------------------------------------------
Districts: 11

Districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

Dates:
['2017-01-01']

Crop types:
crop_type
vegetable           319
fruit               231
herbs               198
spices              121
industrial_crops     88
cash_crops      

In [ ]:
# ======================================================================
# STEP 50G.9 — CROP 2023 DISTRICT DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("CROP 2023 DISTRICT DIAGNOSTIC")
print("=" * 70)

print("\nDistricts:")
print(sorted(crop_2023["district"].unique()))

print("\nRows:")
print(crop_2023[["district"]].to_string(index=False))

print("\nDuplicate districts:")
print(
    crop_2023[
        crop_2023["district"].duplicated(keep=False)
    ]["district"]
)


CROP 2023 DISTRICT DIAGNOSTIC

Districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh', 'Total']

Rows:
         district
          Bentong
             Bera
Cameron Highlands
         Jerantut
          Kuantan
            Lipis
            Maran
            Pekan
             Raub
           Rompin
         Temerloh
            Total

Duplicate districts:
Series([], Name: district, dtype: str)


In [ ]:
# ======================================================================
# STEP 50G.10 — CROP 2023 CLEANING + FINAL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("CROP 2023 CLEANING")
print("=" * 70)

before = len(crop_2023)

# Remove aggregate Total row
crop_2023 = crop_2023[
    crop_2023["district"] != "Total"
].copy()

# Normalize district name
crop_2023["district"] = crop_2023["district"].replace({
    "Lipis": "Kuala Lipis"
})

print("Rows before:", before)
print("Rows after:", len(crop_2023))
print("Rows removed:", before - len(crop_2023))

print("\nFinal districts:")
print(sorted(crop_2023["district"].unique()))

# ======================================================================
# FINAL CROP VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("CROP FINAL VALIDATION")
print("=" * 70)

print("\n[1] Crop 2017–2022 missing values:")
print(crop_2017_2022.isna().sum())

print("\n[2] Crop 2017–2022 exact duplicates:")
print(crop_2017_2022.duplicated().sum())

print("\n[3] Crop 2017–2022 negative production:")
print(
    (crop_2017_2022["production_tonnes"] < 0).sum()
)

print("\n[4] Crop 2023 missing values:")
print(crop_2023.isna().sum())

print("\n[5] Crop 2023 exact duplicates:")
print(crop_2023.duplicated().sum())

print("\n[6] Crop 2023 negative production:")

crop_2023_numeric = crop_2023.select_dtypes(
    include="number"
).columns

for col in crop_2023_numeric:
    print(
        col,
        ":",
        (crop_2023[col] < 0).sum()
    )

print("\n[7] District counts:")
print(
    "2017–2022:",
    crop_2017_2022["district"].nunique()
)

print(
    "2023:",
    crop_2023["district"].nunique()
)

print("\n[8] Final Crop 2023 districts:")
print(sorted(crop_2023["district"].unique()))

print("\n" + "=" * 70)
print("CROP VALIDATION COMPLETE")
print("=" * 70)



CROP 2023 CLEANING
Rows before: 11
Rows after: 11
Rows removed: 0

Final districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuala Lipis', 'Kuantan', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

CROP FINAL VALIDATION

[1] Crop 2017–2022 missing values:
date                 0
state                0
district             0
crop_type            0
crop_species         0
production_tonnes    0
dtype: int64

[2] Crop 2017–2022 exact duplicates:
0

[3] Crop 2017–2022 negative production:
0

[4] Crop 2023 missing values:
district                       0
paddy_production_tonnes        0
oil_palm_production_tonnes     0
rubber_production_tonnes       0
fruits_production_tonnes       0
vegetable_production_tonnes    0
kenaf_production_tonnes        0
cocoa_production_tonnes        0
pepper_production_tonnes       0
pineapple_production_tonnes    0
others_production_tonnes       0
dtype: int64

[5] Crop 2023 exact duplicates:
0

[6] Crop 2023 negative production:
rubber_production

In [ ]:
print([x for x in globals() if "agri" in x.lower() or "employment" in x.lower()])

['agri_employed', 'agri_clean', 'unmapped_agri', 'agri_crosswalk']


In [ ]:
# ======================================================================
# STEP 50H — AGRICULTURAL EMPLOYMENT FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50H — AGRICULTURAL EMPLOYMENT FULL VALIDATION")
print("=" * 70)


# ----------------------------------------------------------------------
# 50H.1 — STRUCTURE
# ----------------------------------------------------------------------

print("\n[50H.1] DATA STRUCTURE")

print("Rows:", len(agri_employed))
print("Columns:", agri_employed.columns.tolist())

print("\nData types:")
print(agri_employed.dtypes)


# ----------------------------------------------------------------------
# 50H.2 — SUMMARY
# ----------------------------------------------------------------------

print("\n[50H.2] DATA SUMMARY")

print(agri_employed.describe(include="all").T)


# ----------------------------------------------------------------------
# 50H.3 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50H.3] MISSING VALUES")

print(agri_employed.isna().sum())


# ----------------------------------------------------------------------
# 50H.4 — EXACT DUPLICATES
# ----------------------------------------------------------------------

print("\n[50H.4] EXACT DUPLICATES")

print(
    "Exact duplicate rows:",
    agri_employed.duplicated().sum()
)


# ----------------------------------------------------------------------
# 50H.5 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50H.5] DISTRICT COVERAGE")

print(
    "Number of districts:",
    agri_employed["district"].nunique()
)

print("\nDistricts:")
print(
    sorted(
        agri_employed["district"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# 50H.6 — YEAR / DATE CHECK
# ----------------------------------------------------------------------

print("\n[50H.6] YEAR / DATE CHECK")

if "year" in agri_employed.columns:
    print("Years:")
    print(sorted(agri_employed["year"].unique()))

elif "date" in agri_employed.columns:
    print("Dates:")
    print(sorted(agri_employed["date"].unique()))

else:
    print("No year/date column present.")
    print("Agricultural employment will require an explicit temporal assignment later.")


# ----------------------------------------------------------------------
# 50H.7 — NUMERIC VARIABLES
# ----------------------------------------------------------------------

print("\n[50H.7] NUMERIC VARIABLES")

numeric_cols = agri_employed.select_dtypes(
    include="number"
).columns.tolist()

print("Numeric columns:", numeric_cols)

for col in numeric_cols:

    print(f"\n{col}")

    print(
        "Minimum:",
        agri_employed[col].min()
    )

    print(
        "Maximum:",
        agri_employed[col].max()
    )

    print(
        "Negative values:",
        (agri_employed[col] < 0).sum()
    )


# ----------------------------------------------------------------------
# 50H.8 — CATEGORICAL VARIABLES
# ----------------------------------------------------------------------

print("\n[50H.8] CATEGORICAL VARIABLES")

for col in agri_employed.select_dtypes(
    exclude="number"
).columns:

    print(f"\n{col}:")
    print(
        agri_employed[col]
        .value_counts(dropna=False)
        .head(20)
    )


# ----------------------------------------------------------------------
# 50H.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(agri_employed))
print(
    "Districts:",
    agri_employed["district"].nunique()
)


STEP 50H — AGRICULTURAL EMPLOYMENT FULL VALIDATION

[50H.1] DATA STRUCTURE
Rows: 12
Columns: ['district', 'total_employment', 'avg_annual_salary_rm']

Data types:
district                str
total_employment        str
avg_annual_salary_rm    str
dtype: object

[50H.2] DATA SUMMARY
                     count unique      top freq
district                12     12  Bentong    1
total_employment        12     12   10,985    1
avg_annual_salary_rm    12     12   20,780    1

[50H.3] MISSING VALUES
district                0
total_employment        0
avg_annual_salary_rm    0
dtype: int64

[50H.4] EXACT DUPLICATES
Exact duplicate rows: 0

[50H.5] DISTRICT COVERAGE
Number of districts: 12

Districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh', 'Total']

[50H.6] YEAR / DATE CHECK
No year/date column present.
Agricultural employment will require an explicit temporal assignment later.

[50H.7] NUMERIC VARIABLES
Numeric

In [ ]:
# ======================================================================
# STEP 50H.10 — AGRICULTURAL EMPLOYMENT DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT DIAGNOSTIC")
print("=" * 70)

print("\n[1] Districts:")
print(sorted(agri_employed["district"].unique()))

print("\n[2] Full records:")
print(agri_employed.to_string(index=False))

print("\n[3] Employment values:")
print(agri_employed["total_employment"].tolist())

print("\n[4] Salary values:")
print(agri_employed["avg_annual_salary_rm"].tolist())

print("\n" + "=" * 70)


AGRICULTURAL EMPLOYMENT DIAGNOSTIC

[1] Districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh', 'Total']

[2] Full records:
         district total_employment avg_annual_salary_rm
          Bentong           10,985               20,780
Cameron Highlands           19,875               25,381
         Jerantut           13,705               22,261
          Kuantan           21,817               19,297
            Lipis           10,447               21,056
            Pekan           19,910               20,481
             Raub           16,430               24,505
         Temerloh           12,292               20,912
           Rompin           30,976               17,946
            Maran           19,319               20.651
             Bera           17,460               20,115
            Total          193,216               20,987

[3] Employment values:
['10,985', '19,875', '13,705', '21,817', '10,44

In [ ]:
# ======================================================================
# STEP 50H.11 — AGRICULTURAL EMPLOYMENT CLEANING
# ======================================================================

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT CLEANING")
print("=" * 70)

# Keep a raw copy before cleaning
agri_employed_raw = agri_employed.copy()

# ----------------------------------------------------------------------
# Remove aggregate
# ----------------------------------------------------------------------

agri_employed = agri_employed[
    agri_employed["district"] != "Total"
].copy()

# ----------------------------------------------------------------------
# Normalize district name
# ----------------------------------------------------------------------

agri_employed["district"] = agri_employed["district"].replace({
    "Lipis": "Kuala Lipis"
})

# ----------------------------------------------------------------------
# Convert employment to numeric
# ----------------------------------------------------------------------

agri_employed["total_employment"] = (
    agri_employed["total_employment"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# ----------------------------------------------------------------------
# Convert salary to numeric
#
# Source contains "20.651", which is inconsistent with the
# comma-separated salary format. Treat as RM 20,651.
# ----------------------------------------------------------------------

agri_employed["avg_annual_salary_rm"] = (
    agri_employed["avg_annual_salary_rm"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("20.651", "20651", regex=False)
    .astype(float)
)

# ----------------------------------------------------------------------
# Display result
# ----------------------------------------------------------------------

print("\nCleaned data:")
print(agri_employed.to_string(index=False))


AGRICULTURAL EMPLOYMENT CLEANING

Cleaned data:
         district  total_employment  avg_annual_salary_rm
          Bentong           10985.0               20780.0
Cameron Highlands           19875.0               25381.0
         Jerantut           13705.0               22261.0
          Kuantan           21817.0               19297.0
      Kuala Lipis           10447.0               21056.0
            Pekan           19910.0               20481.0
             Raub           16430.0               24505.0
         Temerloh           12292.0               20912.0
           Rompin           30976.0               17946.0
            Maran           19319.0               20651.0
             Bera           17460.0               20115.0


In [ ]:
# ======================================================================
# STEP 50H.12 — AGRICULTURAL EMPLOYMENT FINAL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT FINAL VALIDATION")
print("=" * 70)

print("\n[1] Missing values:")
print(agri_employed.isna().sum())

print("\n[2] Exact duplicates:")
print(agri_employed.duplicated().sum())

print("\n[3] District count:")
print(agri_employed["district"].nunique())

print("\n[4] Districts:")
print(sorted(agri_employed["district"].unique()))

print("\n[5] Data types:")
print(agri_employed.dtypes)

print("\n[6] Negative employment:")
print(
    (agri_employed["total_employment"] < 0).sum()
)

print("\n[7] Negative salary:")
print(
    (agri_employed["avg_annual_salary_rm"] < 0).sum()
)

print("\n[8] Employment range:")
print(
    agri_employed["total_employment"].min(),
    "to",
    agri_employed["total_employment"].max()
)

print("\n[9] Salary range:")
print(
    agri_employed["avg_annual_salary_rm"].min(),
    "to",
    agri_employed["avg_annual_salary_rm"].max()
)

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT VALIDATION COMPLETE")
print("=" * 70)


AGRICULTURAL EMPLOYMENT FINAL VALIDATION

[1] Missing values:
district                0
total_employment        0
avg_annual_salary_rm    0
dtype: int64

[2] Exact duplicates:
0

[3] District count:
11

[4] Districts:
['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuala Lipis', 'Kuantan', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']

[5] Data types:
district                    str
total_employment        float64
avg_annual_salary_rm    float64
dtype: object

[6] Negative employment:
0

[7] Negative salary:
0

[8] Employment range:
10447.0 to 30976.0

[9] Salary range:
17946.0 to 25381.0

AGRICULTURAL EMPLOYMENT VALIDATION COMPLETE


In [ ]:
print([x for x in globals() if "inflation" in x.lower()])

['inflation', 'inflation_clean']


In [ ]:
# ======================================================================
# STEP 50I — CPI / INFLATION FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50I — CPI / INFLATION FULL VALIDATION")
print("=" * 70)


# ----------------------------------------------------------------------
# 50I.1 — STRUCTURE
# ----------------------------------------------------------------------

print("\n[50I.1] DATA STRUCTURE")

print("Rows:", len(inflation))
print("Columns:", inflation.columns.tolist())

print("\nData types:")
print(inflation.dtypes)


# ----------------------------------------------------------------------
# 50I.2 — SUMMARY
# ----------------------------------------------------------------------

print("\n[50I.2] DATA SUMMARY")

print(inflation.describe(include="all").T)


# ----------------------------------------------------------------------
# 50I.3 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50I.3] MISSING VALUES")

print(inflation.isna().sum())


# ----------------------------------------------------------------------
# 50I.4 — EXACT DUPLICATES
# ----------------------------------------------------------------------

print("\n[50I.4] EXACT DUPLICATES")

print(
    "Exact duplicate rows:",
    inflation.duplicated().sum()
)


# ----------------------------------------------------------------------
# 50I.5 — DISTRICT COVERAGE
# ----------------------------------------------------------------------

print("\n[50I.5] DISTRICT COVERAGE")

print(
    "Number of districts:",
    inflation["district"].nunique()
)

print("\nDistricts:")
print(
    sorted(
        inflation["district"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# 50I.6 — DATE / YEAR COVERAGE
# ----------------------------------------------------------------------

print("\n[50I.6] DATE / YEAR COVERAGE")

if "year" in inflation.columns:

    print("Years:")
    print(sorted(inflation["year"].unique()))

elif "date" in inflation.columns:

    print("Dates:")
    print(sorted(inflation["date"].unique()))

else:

    print("No year/date column found.")


# ----------------------------------------------------------------------
# 50I.7 — NUMERIC VARIABLES
# ----------------------------------------------------------------------

print("\n[50I.7] NUMERIC VARIABLES")

numeric_cols = inflation.select_dtypes(
    include="number"
).columns.tolist()

print("Numeric columns:", numeric_cols)

for col in numeric_cols:

    print(f"\n{col}")

    print("Minimum:",
          inflation[col].min())

    print("Maximum:",
          inflation[col].max())

    print("Negative values:",
          (inflation[col] < 0).sum())


# ----------------------------------------------------------------------
# 50I.8 — CATEGORICAL VARIABLES
# ----------------------------------------------------------------------

print("\n[50I.8] CATEGORICAL VARIABLES")

for col in inflation.select_dtypes(
    exclude="number"
).columns:

    print(f"\n{col}:")
    print(
        inflation[col]
        .value_counts(dropna=False)
        .head(20)
    )


# ----------------------------------------------------------------------
# 50I.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CPI / INFLATION VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(inflation))

if "district" in inflation.columns:
    print(
        "Districts:",
        inflation["district"].nunique()
    )


STEP 50I — CPI / INFLATION FULL VALIDATION

[50I.1] DATA STRUCTURE
Rows: 2772
Columns: ['state', 'date_by_month', 'division', 'inflation_yoy', 'inflation_mom']

Data types:
state                str
date_by_month        str
division             str
inflation_yoy    float64
inflation_mom    float64
dtype: object

[50I.2] DATA SUMMARY
                count unique         top  freq      mean       std   min  25%  \
state            2772      1      Pahang  2772       NaN       NaN   NaN  NaN   
date_by_month    2772    198  2010-02-01    14       NaN       NaN   NaN  NaN   
division         2772     14     overall   198       NaN       NaN   NaN  NaN   
inflation_yoy  2618.0    NaN         NaN   NaN  1.623224  3.342418 -25.3  0.1   
inflation_mom  2772.0    NaN         NaN   NaN  0.135209  0.938061 -16.4  0.0   

               50%  75%   max  
state          NaN  NaN   NaN  
date_by_month  NaN  NaN   NaN  
division       NaN  NaN   NaN  
inflation_yoy  1.3  2.6  32.7  
inflation_mom  0.0

KeyError: 'district'

In [ ]:
print(inflation.columns.tolist())

['state', 'date_by_month', 'division', 'inflation_yoy', 'inflation_mom']


In [ ]:
# ======================================================================
# STEP 50I — CPI / INFLATION FULL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 50I — CPI / INFLATION FULL VALIDATION")
print("=" * 70)


# ----------------------------------------------------------------------
# 50I.1 — STRUCTURE
# ----------------------------------------------------------------------

print("\n[50I.1] DATA STRUCTURE")

print("Rows:", len(inflation))
print("Columns:", inflation.columns.tolist())

print("\nData types:")
print(inflation.dtypes)


# ----------------------------------------------------------------------
# 50I.2 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[50I.2] MISSING VALUES")

print(inflation.isna().sum())


# ----------------------------------------------------------------------
# 50I.3 — EXACT DUPLICATES
# ----------------------------------------------------------------------

print("\n[50I.3] EXACT DUPLICATES")

print(
    "Exact duplicate rows:",
    inflation.duplicated().sum()
)


# ----------------------------------------------------------------------
# 50I.4 — DATE COVERAGE
# ----------------------------------------------------------------------

print("\n[50I.4] DATE COVERAGE")

print("Number of dates:",
      inflation["date_by_month"].nunique())

print("\nFirst date:",
      inflation["date_by_month"].min())

print("Last date:",
      inflation["date_by_month"].max())


# ----------------------------------------------------------------------
# 50I.5 — DIVISION COVERAGE
# ----------------------------------------------------------------------

print("\n[50I.5] DIVISION COVERAGE")

print(
    "Number of divisions:",
    inflation["division"].nunique()
)

print("\nDivisions:")
print(
    sorted(
        inflation["division"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# 50I.6 — RECORDS PER DIVISION
# ----------------------------------------------------------------------

print("\n[50I.6] RECORDS PER DIVISION")

print(
    inflation["division"]
    .value_counts()
    .sort_index()
)


# ----------------------------------------------------------------------
# 50I.7 — INFLATION RANGE
# ----------------------------------------------------------------------

print("\n[50I.7] INFLATION RANGE")

for col in [
    "inflation_yoy",
    "inflation_mom"
]:

    print(f"\n{col}")

    print(
        "Minimum:",
        inflation[col].min()
    )

    print(
        "Maximum:",
        inflation[col].max()
    )

    print(
        "Missing:",
        inflation[col].isna().sum()
    )


# ----------------------------------------------------------------------
# 50I.8 — DUPLICATE MONTH-DIVISION RECORDS
# ----------------------------------------------------------------------

print("\n[50I.8] DUPLICATE MONTH-DIVISION CHECK")

duplicate_cpi = inflation[
    inflation.duplicated(
        subset=["date_by_month", "division"],
        keep=False
    )
]

print(
    "Duplicate date-division records:",
    len(duplicate_cpi)
)


# ----------------------------------------------------------------------
# 50I.9 — FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CPI / INFLATION VALIDATION COMPLETE")
print("=" * 70)

print("Rows:", len(inflation))

print(
    "Dates:",
    inflation["date_by_month"].nunique()
)

print(
    "Divisions:",
    inflation["division"].nunique()
)


STEP 50I — CPI / INFLATION FULL VALIDATION

[50I.1] DATA STRUCTURE
Rows: 2772
Columns: ['state', 'date_by_month', 'division', 'inflation_yoy', 'inflation_mom']

Data types:
state                str
date_by_month        str
division             str
inflation_yoy    float64
inflation_mom    float64
dtype: object

[50I.2] MISSING VALUES
state              0
date_by_month      0
division           0
inflation_yoy    154
inflation_mom      0
dtype: int64

[50I.3] EXACT DUPLICATES
Exact duplicate rows: 0

[50I.4] DATE COVERAGE
Number of dates: 198

First date: 2010-02-01
Last date: 2026-07-01

[50I.5] DIVISION COVERAGE
Number of divisions: 14

Divisions:
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', 'overall']

[50I.6] RECORDS PER DIVISION
division
01         198
02         198
03         198
04         198
05         198
06         198
07         198
08         198
09         198
10         198
11         198
12         198
13         198
overall    198
Nam

In [ ]:
# ======================================================================
# STEP 50I.10 — CPI YoY MISSINGNESS DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("CPI YoY MISSINGNESS DIAGNOSTIC")
print("=" * 70)

# Missing YoY by division
print("\n[1] Missing YoY by division:")
print(
    inflation[
        inflation["inflation_yoy"].isna()
    ]["division"].value_counts().sort_index()
)

# Missing YoY by month
print("\n[2] Missing YoY by date:")
print(
    inflation[
        inflation["inflation_yoy"].isna()
    ]["date_by_month"].value_counts().sort_index().head(20)
)

# Duplicate date-division check
print("\n[3] Duplicate date-division records:")
print(
    inflation.duplicated(
        subset=["date_by_month", "division"],
        keep=False
    ).sum()
)

# Division list
print("\n[4] Divisions:")
print(sorted(inflation["division"].unique()))

print("\n" + "=" * 70)


CPI YoY MISSINGNESS DIAGNOSTIC

[1] Missing YoY by division:
division
01         11
02         11
03         11
04         11
05         11
06         11
07         11
08         11
09         11
10         11
11         11
12         11
13         11
overall    11
Name: count, dtype: int64

[2] Missing YoY by date:
date_by_month
2010-02-01    14
2010-03-01    14
2010-04-01    14
2010-05-01    14
2010-06-01    14
2010-07-01    14
2010-08-01    14
2010-09-01    14
2010-10-01    14
2010-11-01    14
2010-12-01    14
Name: count, dtype: int64

[3] Duplicate date-division records:
0

[4] Divisions:
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', 'overall']



# CROSS-DATASET VALIDATION 

In [ ]:
# ======================================================================
# STEP 51 — CROSS-DATASET COVERAGE VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 51 — CROSS-DATASET COVERAGE VALIDATION")
print("=" * 70)


# ----------------------------------------------------------------------
# 51.1 — MASTER DISTRICT LIST
# ----------------------------------------------------------------------

print("\n[51.1] DISTRICT COVERAGE")

datasets = {
    "Labour": labour,
    "Income": income,
    "Poverty": poverty,
    "Population": population,
    "Amenities": amenities,
    "Enrollment": enrollment,
    "Crop 2017-2022": crop_2017_2022,
    "Crop 2023": crop_2023,
    "Agricultural Employment": agri_employed
}

for name, df in datasets.items():

    if "district" in df.columns:

        districts = set(
            df["district"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

        print(
            f"{name:25s}: "
            f"{len(districts):2d} districts"
        )

        print(
            "   ",
            sorted(districts)
        )


# ----------------------------------------------------------------------
# 51.2 — MASTER DISTRICT COMPARISON
# ----------------------------------------------------------------------

print("\n[51.2] DISTRICTS MISSING FROM 11-DISTRICT MASTER")

master_districts = set(
    income["district"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Master districts:")
print(sorted(master_districts))

for name, df in datasets.items():

    if "district" in df.columns:

        districts = set(
            df["district"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

        missing_from_dataset = master_districts - districts
        extra_in_dataset = districts - master_districts

        print(f"\n{name}")

        print(
            "Missing master districts:",
            sorted(missing_from_dataset)
        )

        print(
            "Extra districts:",
            sorted(extra_in_dataset)
        )


# ----------------------------------------------------------------------
# 51.3 — TEMPORAL COVERAGE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[51.3] TEMPORAL COVERAGE")
print("=" * 70)


def show_year_or_date(name, df):

    if "year" in df.columns:

        print(
            f"{name:25s}: "
            f"{sorted(df['year'].dropna().unique())}"
        )

    elif "date" in df.columns:

        dates = pd.to_datetime(
            df["date"],
            errors="coerce"
        )

        print(
            f"{name:25s}: "
            f"{dates.min().date()} → {dates.max().date()}"
        )

    elif "date_by_month" in df.columns:

        dates = pd.to_datetime(
            df["date_by_month"],
            errors="coerce"
        )

        print(
            f"{name:25s}: "
            f"{dates.min().date()} → {dates.max().date()}"
        )

    else:

        print(
            f"{name:25s}: no temporal column"
        )


for name, df in {
    "Labour": labour,
    "Income": income,
    "Poverty": poverty,
    "Population": population,
    "Amenities": amenities,
    "Enrollment": enrollment,
    "Crop 2017-2022": crop_2017_2022,
    "Crop 2023": crop_2023,
    "Agricultural Employment": agri_employed,
    "CPI": inflation
}.items():

    show_year_or_date(name, df)


print("\n" + "=" * 70)
print("STEP 51 COVERAGE CHECK COMPLETE")
print("=" * 70)


STEP 51 — CROSS-DATASET COVERAGE VALIDATION

[51.1] DISTRICT COVERAGE
Labour                   : 11 districts
    ['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']
Income                   : 11 districts
    ['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']
Poverty                  : 11 districts
    ['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']
Population               : 12 districts
    ['Bentong', 'Bera', 'Cameron Highland', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']
Amenities                : 11 districts
    ['Bentong', 'Bera', 'Cameron Highlands', 'Jerantut', 'Kuantan', 'Lipis', 'Maran', 'Pekan', 'Raub', 'Rompin', 'Temerloh']
Enrollment               : 11 districts
    ['Bentong', 'Bera', 'Cameron Highlands',

In [ ]:
# ======================================================================
# STEP 51.2 — STANDARDIZE DISTRICT NAMES ACROSS DATASETS
# ======================================================================

print("\n" + "=" * 70)
print("STEP 51.2 — DISTRICT NAME STANDARDIZATION")
print("=" * 70)

district_mapping = {
    "Lipis": "Kuala Lipis",
    "Cameron Highland": "Cameron Highlands"
}

datasets_to_standardize = {
    "labour": labour,
    "income": income,
    "poverty": poverty,
    "population": population,
    "amenities": amenities,
    "enrollment": enrollment,
    "crop_2017_2022": crop_2017_2022,
    "crop_2023": crop_2023,
    "agri_employed": agri_employed
}

for name, df in datasets_to_standardize.items():

    if "district" in df.columns:

        df["district"] = (
            df["district"]
            .astype(str)
            .str.strip()
            .replace(district_mapping)
        )

        print(
            f"{name:20s}: "
            f"{df['district'].nunique()} districts"
        )


# ----------------------------------------------------------------------
# MASTER DISTRICT LIST
# ----------------------------------------------------------------------

master_districts = sorted(
    income["district"].unique()
)

print("\nMASTER DISTRICT LIST:")
for i, district in enumerate(master_districts, 1):
    print(f"{i:2d}. {district}")

print("\nNumber of master districts:",
      len(master_districts))


STEP 51.2 — DISTRICT NAME STANDARDIZATION
labour              : 11 districts
income              : 11 districts
poverty             : 11 districts
population          : 11 districts
amenities           : 11 districts
enrollment          : 11 districts
crop_2017_2022      : 11 districts
crop_2023           : 11 districts
agri_employed       : 11 districts

MASTER DISTRICT LIST:
 1. Bentong
 2. Bera
 3. Cameron Highlands
 4. Jerantut
 5. Kuala Lipis
 6. Kuantan
 7. Maran
 8. Pekan
 9. Raub
10. Rompin
11. Temerloh

Number of master districts: 11


In [ ]:
# ======================================================================
# STEP 51.3 — VERIFY DISTRICT ALIGNMENT
# ======================================================================

print("\n" + "=" * 70)
print("STEP 51.3 — DISTRICT ALIGNMENT VERIFICATION")
print("=" * 70)

master_districts = set(income["district"].unique())

for name, df in datasets_to_standardize.items():

    districts = set(df["district"].dropna().unique())

    missing = master_districts - districts
    extra = districts - master_districts

    print(f"\n{name}")
    print("  District count:", len(districts))
    print("  Missing:", sorted(missing))
    print("  Extra:", sorted(extra))

print("\n" + "=" * 70)


STEP 51.3 — DISTRICT ALIGNMENT VERIFICATION

labour
  District count: 11
  Missing: []
  Extra: []

income
  District count: 11
  Missing: []
  Extra: []

poverty
  District count: 11
  Missing: []
  Extra: []

population
  District count: 11
  Missing: []
  Extra: []

amenities
  District count: 11
  Missing: []
  Extra: []

enrollment
  District count: 11
  Missing: []
  Extra: []

crop_2017_2022
  District count: 11
  Missing: []
  Extra: []

crop_2023
  District count: 11
  Missing: []
  Extra: []

agri_employed
  District count: 11
  Missing: []
  Extra: []



# TEMPORAL ALIGNMENT

In [ ]:
# ======================================================================
# STEP 52 — TEMPORAL ALIGNMENT CHECK
# ======================================================================

print("\n" + "=" * 70)
print("STEP 52 — TEMPORAL ALIGNMENT CHECK")
print("=" * 70)


# ----------------------------------------------------------------------
# Labour
# ----------------------------------------------------------------------

print("\n[LABOUR]")
print(
    sorted(
        labour["year"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# Income
# ----------------------------------------------------------------------

print("\n[INCOME]")
print(
    sorted(
        income["year"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# Poverty
# ----------------------------------------------------------------------

print("\n[POVERTY]")
print(
    sorted(
        poverty["year"].dropna().unique()
    )
)


# ----------------------------------------------------------------------
# Population
# ----------------------------------------------------------------------

print("\n[POPULATION]")

population_dates = pd.to_datetime(
    population["date"],
    errors="coerce"
)

print(
    "Years:",
    sorted(
        population_dates.dt.year.unique()
    )
)


# ----------------------------------------------------------------------
# Amenities
# ----------------------------------------------------------------------

print("\n[AMENITIES]")

amenities_dates = pd.to_datetime(
    amenities["date"],
    errors="coerce"
)

print(
    "Years:",
    sorted(
        amenities_dates.dt.year.unique()
    )
)


# ----------------------------------------------------------------------
# Enrollment
# ----------------------------------------------------------------------

print("\n[ENROLLMENT]")

enrollment_dates = pd.to_datetime(
    enrollment["date"],
    errors="coerce"
)

print(
    "Years:",
    sorted(
        enrollment_dates.dt.year.unique()
    )
)


# ----------------------------------------------------------------------
# Crop 2017–2022
# ----------------------------------------------------------------------

print("\n[CROP 2017–2022]")

crop_dates = pd.to_datetime(
    crop_2017_2022["date"],
    errors="coerce"
)

print(
    "Years:",
    sorted(
        crop_dates.dt.year.unique()
    )
)


# ----------------------------------------------------------------------
# Crop 2023
# ----------------------------------------------------------------------

print("\n[CROP 2023]")

print(
    "Analytical year: 2023"
)


# ----------------------------------------------------------------------
# Agricultural Employment
# ----------------------------------------------------------------------

print("\n[AGRICULTURAL EMPLOYMENT]")

print(
    "Analytical year: 2023"
)


# ----------------------------------------------------------------------
# CPI
# ----------------------------------------------------------------------

print("\n[CPI]")

cpi_dates = pd.to_datetime(
    inflation["date_by_month"],
    errors="coerce"
)

print(
    "Years:",
    sorted(
        cpi_dates.dt.year.unique()
    )
)


print("\n" + "=" * 70)
print("STEP 52 TEMPORAL CHECK COMPLETE")
print("=" * 70)


STEP 52 — TEMPORAL ALIGNMENT CHECK

[LABOUR]
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

[INCOME]
[np.int64(2019), np.int64(2022), np.int64(2024)]

[POVERTY]
[np.int64(2019), np.int64(2022), np.int64(2024)]

[POPULATION]
Years: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

[AMENITIES]
Years: [np.int32(2016), np.int32(2019), np.int32(2022), np.int32(2024)]

[ENROLLMENT]
Years: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

[CROP 2017–2022]
Years: [np.int32(2017)]

[CROP 2023]
Analytical year: 2023

[AGRICULTURAL EMPLOYMENT]
Analytical year: 2023

[CPI]
Years: [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.i

In [ ]:
# ======================================================================
# STEP 53 — CONVERT DATASETS TO DISTRICT × YEAR
# ======================================================================

print("\n" + "=" * 70)
print("STEP 53 — DISTRICT-YEAR AGGREGATION")
print("=" * 70)


# ======================================================================
# 53.1 — LABOUR
# ======================================================================

labour_annual = labour.copy()

print("\n[53.1] LABOUR")
print("Rows:", len(labour_annual))
print("Districts:", labour_annual["district"].nunique())
print("Years:", sorted(labour_annual["year"].unique()))


# ======================================================================
# 53.2 — INCOME
# ======================================================================

income_annual = income.copy()

print("\n[53.2] INCOME")
print("Rows:", len(income_annual))
print("Years:", sorted(income_annual["year"].unique()))


# ======================================================================
# 53.3 — POVERTY
# ======================================================================

poverty_annual = poverty.copy()

print("\n[53.3] POVERTY")
print("Rows:", len(poverty_annual))
print("Years:", sorted(poverty_annual["year"].unique()))


# ======================================================================
# 53.4 — POPULATION
# ======================================================================

population_annual = population.copy()

population_annual["year"] = pd.to_datetime(
    population_annual["date"]
).dt.year

population_annual = (
    population_annual
    .groupby(
        ["district", "year"],
        as_index=False
    )["population_000"]
    .sum()
)

print("\n[53.4] POPULATION")
print("Rows:", len(population_annual))
print("Districts:", population_annual["district"].nunique())
print("Years:", sorted(population_annual["year"].unique()))


# ======================================================================
# 53.5 — AMENITIES
# ======================================================================

amenities_annual = amenities.copy()

amenities_annual["year"] = pd.to_datetime(
    amenities_annual["date"]
).dt.year

amenities_annual = (
    amenities_annual[
        [
            "district",
            "year",
            "piped_water_%",
            "sanitation_%",
            "electricity_%"
        ]
    ]
    .copy()
)

print("\n[53.5] AMENITIES")
print("Rows:", len(amenities_annual))
print("Years:", sorted(amenities_annual["year"].unique()))


# ======================================================================
# 53.6 — ENROLLMENT
# ======================================================================

enrollment_annual = enrollment.copy()

enrollment_annual["year"] = pd.to_datetime(
    enrollment_annual["date"]
).dt.year

enrollment_annual = (
    enrollment_annual
    .groupby(
        ["district", "year"],
        as_index=False
    )["students"]
    .sum()
    .rename(
        columns={
            "students": "students_total"
        }
    )
)

print("\n[53.6] ENROLLMENT")
print("Rows:", len(enrollment_annual))
print("Districts:", enrollment_annual["district"].nunique())
print("Years:", sorted(enrollment_annual["year"].unique()))


# ======================================================================
# 53.7 — CROP 2017
# ======================================================================

crop_annual = crop_2017_2022.copy()

crop_annual["year"] = pd.to_datetime(
    crop_annual["date"]
).dt.year

crop_annual = (
    crop_annual
    .groupby(
        ["district", "year"],
        as_index=False
    )["production_tonnes"]
    .sum()
)

print("\n[53.7] CROP 2017")
print("Rows:", len(crop_annual))
print("Years:", sorted(crop_annual["year"].unique()))


# ======================================================================
# STEP 53.8 — CROP 2023 NUMERIC CONVERSION + AGGREGATION
# ======================================================================

print("\n" + "=" * 70)
print("STEP 53.8 — CROP 2023 AGGREGATION")
print("=" * 70)

crop_2023_annual = crop_2023.copy()

# Assign analytical year
crop_2023_annual["year"] = 2023

# Identify crop production columns
crop_2023_numeric = [
    col for col in crop_2023_annual.columns
    if col.endswith("_production_tonnes")
]

print("\nProduction columns:")
print(crop_2023_numeric)

# Convert production columns to numeric
for col in crop_2023_numeric:
    crop_2023_annual[col] = (
        crop_2023_annual[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(["nan", "None", ""], pd.NA)
    )

    crop_2023_annual[col] = pd.to_numeric(
        crop_2023_annual[col],
        errors="coerce"
    )

# Calculate total crop production
crop_2023_annual["crop_production_total_tonnes"] = (
    crop_2023_annual[crop_2023_numeric]
    .sum(axis=1, min_count=1)
)

# Keep only required columns
crop_2023_annual = crop_2023_annual[
    [
        "district",
        "year",
        "crop_production_total_tonnes"
    ]
].copy()

# Rename 2017 crop production
crop_annual = crop_annual.rename(
    columns={
        "production_tonnes":
        "crop_production_total_tonnes"
    }
)

# Combine 2017 + 2023
crop_annual = pd.concat(
    [
        crop_annual,
        crop_2023_annual
    ],
    ignore_index=True
)

print("\nCrop annual result:")
print(crop_annual.head())

print("\nRows:", len(crop_annual))
print("Districts:", crop_annual["district"].nunique())
print("Years:", sorted(crop_annual["year"].unique()))

# ======================================================================
# STEP 53.9 — AGRICULTURAL EMPLOYMENT
# ======================================================================

agri_annual = agri_employed.copy()

agri_annual["year"] = 2023

print("\n" + "=" * 70)
print("AGRICULTURAL EMPLOYMENT")
print("=" * 70)

print("Rows:", len(agri_annual))
print("Districts:", agri_annual["district"].nunique())
print("Year:", agri_annual["year"].unique())


# ======================================================================
# STEP 53.10 — ANNUAL DATASET SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("STEP 53 — DISTRICT-YEAR AGGREGATION SUMMARY")
print("=" * 70)

annual_datasets = {
    "Labour": labour_annual,
    "Income": income_annual,
    "Poverty": poverty_annual,
    "Population": population_annual,
    "Amenities": amenities_annual,
    "Enrollment": enrollment_annual,
    "Crop": crop_annual,
    "Agricultural Employment": agri_annual
}

for name, df in annual_datasets.items():

    print(
        f"{name:25s} "
        f"rows={len(df):4d} "
        f"districts={df['district'].nunique():2d} "
        f"years={sorted(df['year'].unique())}"
    )


STEP 53 — DISTRICT-YEAR AGGREGATION

[53.1] LABOUR
Rows: 77
Districts: 11
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

[53.2] INCOME
Rows: 33
Years: [np.int64(2019), np.int64(2022), np.int64(2024)]

[53.3] POVERTY
Rows: 33
Years: [np.int64(2019), np.int64(2022), np.int64(2024)]

[53.4] POPULATION
Rows: 66
Districts: 11
Years: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

[53.5] AMENITIES
Rows: 44
Years: [np.int32(2016), np.int32(2019), np.int32(2022), np.int32(2024)]

[53.6] ENROLLMENT
Rows: 99
Districts: 11
Years: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

[53.7] CROP 2017
Rows: 11
Years: [np.int32(2017)]

STEP 53.8 — CROP 2023 AGGREGATION

Production columns:
['paddy_production_tonnes', 'oil_palm_production_tonnes', 'rubber_production_tonnes', 'fruits_prod

# MASTER DISTRICT-YEAR PANEL

In [ ]:
# ======================================================================
# STEP 54 — MASTER DISTRICT-YEAR PANEL SKELETON
# ======================================================================

print("\n" + "=" * 70)
print("STEP 54 — MASTER DISTRICT-YEAR PANEL")
print("=" * 70)

# ----------------------------------------------------------------------
# 54.1 — MASTER DISTRICTS
# ----------------------------------------------------------------------

master_districts = sorted(
    income_annual["district"].unique()
)

# ----------------------------------------------------------------------
# 54.2 — ANALYTICAL YEARS
# ----------------------------------------------------------------------

analysis_years = list(range(2018, 2025))

print("\nMaster districts:", len(master_districts))
print("Analysis years:", analysis_years)

# ----------------------------------------------------------------------
# 54.3 — CREATE DISTRICT × YEAR SKELETON
# ----------------------------------------------------------------------

master_panel = pd.MultiIndex.from_product(
    [
        master_districts,
        analysis_years
    ],
    names=["district", "year"]
).to_frame(index=False)

print("\nMaster skeleton:")
print(master_panel.head(15))

print("\nRows:", len(master_panel))
print(
    "Expected rows:",
    len(master_districts) * len(analysis_years)
)

print(
    "Districts:",
    master_panel["district"].nunique()
)

print(
    "Years:",
    sorted(master_panel["year"].unique())
)

# ----------------------------------------------------------------------
# 54.4 — DUPLICATE CHECK
# ----------------------------------------------------------------------

print("\nDuplicate district-year records:")

print(
    master_panel.duplicated(
        subset=["district", "year"]
    ).sum()
)

print("\n" + "=" * 70)
print("MASTER SKELETON CREATED")
print("=" * 70)


STEP 54 — MASTER DISTRICT-YEAR PANEL

Master districts: 11
Analysis years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]

Master skeleton:
             district  year
0             Bentong  2018
1             Bentong  2019
2             Bentong  2020
3             Bentong  2021
4             Bentong  2022
5             Bentong  2023
6             Bentong  2024
7                Bera  2018
8                Bera  2019
9                Bera  2020
10               Bera  2021
11               Bera  2022
12               Bera  2023
13               Bera  2024
14  Cameron Highlands  2018

Rows: 77
Expected rows: 77
Districts: 11
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Duplicate district-year records:
0

MASTER SKELETON CREATED


# BUILIDING MASTER PANEL

In [ ]:
# ======================================================================
# STEP 55 — BUILD MASTER DISTRICT-YEAR PANEL
# ======================================================================

print("\n" + "=" * 70)
print("STEP 55 — MASTER PANEL CONSTRUCTION")
print("=" * 70)

# Start from the 77-row skeleton
master_panel = master_panel.copy()


# ======================================================================
# 55.1 — LABOUR
# ======================================================================

master_panel = master_panel.merge(
    labour_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("\n[55.1] After Labour:", master_panel.shape)


# ======================================================================
# 55.2 — INCOME
# ======================================================================

master_panel = master_panel.merge(
    income_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.2] After Income:", master_panel.shape)


# ======================================================================
# 55.3 — POVERTY
# ======================================================================

master_panel = master_panel.merge(
    poverty_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.3] After Poverty:", master_panel.shape)


# ======================================================================
# 55.4 — POPULATION
# ======================================================================

master_panel = master_panel.merge(
    population_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.4] After Population:", master_panel.shape)


# ======================================================================
# 55.5 — AMENITIES
# ======================================================================

master_panel = master_panel.merge(
    amenities_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.5] After Amenities:", master_panel.shape)


# ======================================================================
# 55.6 — ENROLLMENT
# ======================================================================

master_panel = master_panel.merge(
    enrollment_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.6] After Enrollment:", master_panel.shape)


# ======================================================================
# 55.7 — CROP
# ======================================================================

master_panel = master_panel.merge(
    crop_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print("[55.7] After Crop:", master_panel.shape)


# ======================================================================
# 55.8 — AGRICULTURAL EMPLOYMENT
# ======================================================================

master_panel = master_panel.merge(
    agri_annual,
    on=["district", "year"],
    how="left",
    validate="one_to_one"
)

print(
    "[55.8] After Agricultural Employment:",
    master_panel.shape
)


# ======================================================================
# 55.9 — FINAL STRUCTURE
# ======================================================================

print("\n" + "=" * 70)
print("MASTER PANEL STRUCTURE")
print("=" * 70)

print("\nShape:", master_panel.shape)

print("\nColumns:")
print(master_panel.columns.tolist())

print("\nDuplicate district-year records:")

print(
    master_panel.duplicated(
        subset=["district", "year"]
    ).sum()
)

print("\n" + "=" * 70)
print("STEP 55 COMPLETE")
print("=" * 70)


STEP 55 — MASTER PANEL CONSTRUCTION

[55.1] After Labour: (77, 9)
[55.2] After Income: (77, 11)
[55.3] After Poverty: (77, 13)
[55.4] After Population: (77, 14)
[55.5] After Amenities: (77, 17)
[55.6] After Enrollment: (77, 18)
[55.7] After Crop: (77, 19)
[55.8] After Agricultural Employment: (77, 21)

MASTER PANEL STRUCTURE

Shape: (77, 21)

Columns:
['district', 'year', 'labour_force_000', 'employed_000', 'unemployed_000', 'outside_lf_000', 'participation_rate_pct', 'unemployment_rate_pct', 'employment_pop_ratio_pct', 'income_mean_rm', 'income_median_rm', 'poverty_absolute_pct', 'poverty_relative_pct', 'population_000', 'piped_water_%', 'sanitation_%', 'electricity_%', 'students_total', 'crop_production_total_tonnes', 'total_employment', 'avg_annual_salary_rm']

Duplicate district-year records:
0

STEP 55 COMPLETE


In [ ]:
# ======================================================================
# STEP 56 — MASTER PANEL MISSINGNESS AUDIT
# ======================================================================

print("\n" + "=" * 70)
print("STEP 56 — MASTER PANEL MISSINGNESS AUDIT")
print("=" * 70)


# ----------------------------------------------------------------------
# 56.1 — MISSING COUNT
# ----------------------------------------------------------------------

print("\n[56.1] MISSING VALUES BY VARIABLE")

missing_count = (
    master_panel.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_count)


# ----------------------------------------------------------------------
# 56.2 — MISSING PERCENTAGE
# ----------------------------------------------------------------------

print("\n[56.2] MISSING PERCENTAGE")

missing_pct = (
    master_panel.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(missing_pct)


# ----------------------------------------------------------------------
# 56.3 — COMBINED MISSINGNESS TABLE
# ----------------------------------------------------------------------

print("\n[56.3] MISSINGNESS SUMMARY")

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
})

print(
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
)


# ----------------------------------------------------------------------
# 56.4 — MISSINGNESS BY YEAR
# ----------------------------------------------------------------------

print("\n[56.4] MISSING VALUES BY YEAR")

print(
    master_panel
    .isna()
    .groupby(master_panel["year"])
    .sum()
)


# ----------------------------------------------------------------------
# 56.5 — MISSINGNESS BY DISTRICT
# ----------------------------------------------------------------------

print("\n[56.5] MISSING VALUES BY DISTRICT")

district_missing = (
    master_panel
    .isna()
    .groupby(master_panel["district"])
    .sum()
)

print(district_missing)


# ----------------------------------------------------------------------
# 56.6 — COMPLETE CASES
# ----------------------------------------------------------------------

print("\n[56.6] COMPLETE DISTRICT-YEAR RECORDS")

complete_rows = master_panel.notna().all(axis=1).sum()

print(
    "Completely complete rows:",
    complete_rows
)

print(
    "Percentage complete:",
    round(
        complete_rows / len(master_panel) * 100,
        2
    ),
    "%"
)


print("\n" + "=" * 70)
print("STEP 56 COMPLETE")
print("=" * 70)


STEP 56 — MASTER PANEL MISSINGNESS AUDIT

[56.1] MISSING VALUES BY VARIABLE
avg_annual_salary_rm            66
crop_production_total_tonnes    66
total_employment                66
poverty_relative_pct            44
piped_water_%                   44
sanitation_%                    44
electricity_%                   44
income_median_rm                44
income_mean_rm                  44
poverty_absolute_pct            44
population_000                  22
district                         0
year                             0
unemployed_000                   0
employed_000                     0
labour_force_000                 0
employment_pop_ratio_pct         0
outside_lf_000                   0
participation_rate_pct           0
unemployment_rate_pct            0
students_total                   0
dtype: int64

[56.2] MISSING PERCENTAGE
avg_annual_salary_rm            85.71
crop_production_total_tonnes    85.71
total_employment                85.71
poverty_relative_pct            57

In [ ]:
# ======================================================================
# STEP 57 — MASTER PANEL MISSINGNESS BY YEAR
# ======================================================================

print("\n" + "=" * 70)
print("STEP 57 — MISSINGNESS BY YEAR")
print("=" * 70)

key_vars = [
    "income_mean_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "population_000",
    "piped_water_%",
    "sanitation_%",
    "electricity_%",
    "students_total",
    "crop_production_total_tonnes",
    "total_employment",
    "avg_annual_salary_rm"
]

for year in sorted(master_panel["year"].unique()):

    subset = master_panel[
        master_panel["year"] == year
    ]

    print(f"\nYEAR {year}")
    print("-" * 50)

    for col in key_vars:

        missing = subset[col].isna().sum()

        print(
            f"{col:35s}: "
            f"{missing:2d}/11 missing"
        )

print("\n" + "=" * 70)


STEP 57 — MISSINGNESS BY YEAR

YEAR 2018
--------------------------------------------------
income_mean_rm                     : 11/11 missing
poverty_absolute_pct               : 11/11 missing
poverty_relative_pct               : 11/11 missing
population_000                     : 11/11 missing
piped_water_%                      : 11/11 missing
sanitation_%                       : 11/11 missing
electricity_%                      : 11/11 missing
students_total                     :  0/11 missing
crop_production_total_tonnes       : 11/11 missing
total_employment                   : 11/11 missing
avg_annual_salary_rm               : 11/11 missing

YEAR 2019
--------------------------------------------------
income_mean_rm                     :  0/11 missing
poverty_absolute_pct               :  0/11 missing
poverty_relative_pct               :  0/11 missing
population_000                     : 11/11 missing
piped_water_%                      :  0/11 missing
sanitation_%                 

In [ ]:
# ======================================================================
# STEP 58 — COMPACT TEMPORAL MISSINGNESS MATRIX
# ======================================================================

key_vars = [
    "labour_force_000",
    "income_mean_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "population_000",
    "piped_water_%",
    "sanitation_%",
    "electricity_%",
    "students_total",
    "crop_production_total_tonnes",
    "total_employment",
    "avg_annual_salary_rm"
]

missing_matrix = (
    master_panel
    .groupby("year")[key_vars]
    .apply(lambda x: x.isna().sum())
)

print("\n" + "=" * 70)
print("STEP 58 — TEMPORAL MISSINGNESS MATRIX")
print("=" * 70)

print(missing_matrix)

print("\n0 = complete for all 11 districts")
print("11 = completely unavailable for that year")


STEP 58 — TEMPORAL MISSINGNESS MATRIX
      labour_force_000  income_mean_rm  poverty_absolute_pct  \
year                                                           
2018                 0              11                    11   
2019                 0               0                     0   
2020                 0              11                    11   
2021                 0              11                    11   
2022                 0               0                     0   
2023                 0              11                    11   
2024                 0               0                     0   

      poverty_relative_pct  population_000  piped_water_%  sanitation_%  \
year                                                                      
2018                    11              11             11            11   
2019                     0              11              0             0   
2020                    11               0             11            11   
2021     

In [ ]:
# ======================================================================
# STEP 59 — TEMPORAL ALIGNMENT FOR CORE DVI INDICATORS
# ======================================================================

print("\n" + "=" * 70)
print("STEP 59 — TEMPORAL ALIGNMENT")
print("=" * 70)

# ----------------------------------------------------------------------
# 59.1 — WORKING PERIOD
# ----------------------------------------------------------------------

dvi_years = list(range(2019, 2025))

dvi_panel = master_panel[
    master_panel["year"].isin(dvi_years)
].copy()

print("\nDVI analytical years:")
print(dvi_years)

print("Rows:", len(dvi_panel))


# ----------------------------------------------------------------------
# 59.2 — VARIABLES TO INTERPOLATE
# ----------------------------------------------------------------------

interpolation_vars = [
    "piped_water_%",
    "sanitation_%",
    "electricity_%"
]


# ----------------------------------------------------------------------
# 59.3 — SORT
# ----------------------------------------------------------------------

dvi_panel = dvi_panel.sort_values(
    ["district", "year"]
).reset_index(drop=True)


# ----------------------------------------------------------------------
# 59.4 — CREATE ORIGINAL OBSERVED FLAGS
# ----------------------------------------------------------------------
# IMPORTANT:
# Create these BEFORE interpolation so that the flags represent
# the original source data, not the interpolated data.

for col in interpolation_vars:

    observed_col = f"{col}_observed"

    dvi_panel[observed_col] = (
        dvi_panel[col].notna()
    )


# ----------------------------------------------------------------------
# 59.5 — LINEAR INTERPOLATION WITHIN DISTRICT
# ----------------------------------------------------------------------

for col in interpolation_vars:

    dvi_panel[col] = (
        dvi_panel
        .groupby("district")[col]
        .transform(
            lambda s: s.interpolate(
                method="linear",
                limit_area="inside"
            )
        )
    )


# ----------------------------------------------------------------------
# 59.6 — REPORT REMAINING MISSING VALUES
# ----------------------------------------------------------------------

print("\nRemaining missing values after interpolation:")

print(
    dvi_panel[
        interpolation_vars
    ].isna().sum()
)


# ----------------------------------------------------------------------
# 59.7 — OBSERVED VS INTERPOLATED COUNTS
# ----------------------------------------------------------------------

print("\nObserved vs interpolated:")

for col in interpolation_vars:

    observed_col = f"{col}_observed"

    observed = dvi_panel[observed_col].sum()

    interpolated = (
        (~dvi_panel[observed_col])
        & dvi_panel[col].notna()
    ).sum()

    print(
        f"{col:25s} "
        f"observed={observed:2d} "
        f"interpolated={interpolated:2d}"
    )


# ----------------------------------------------------------------------
# 59.8 — CHECK 2019–2024 COVERAGE
# ----------------------------------------------------------------------

print("\nDVI panel coverage:")

print(
    dvi_panel
    .groupby("year")
    .size()
)


# ----------------------------------------------------------------------
# FINAL
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 59 COMPLETE")
print("=" * 70)

print(
    "DVI panel shape:",
    dvi_panel.shape
)

print(
    "Districts:",
    dvi_panel["district"].nunique()
)

print(
    "Years:",
    sorted(dvi_panel["year"].unique())
)


STEP 59 — TEMPORAL ALIGNMENT


NameError: name 'master_panel' is not defined

In [ ]:
# ======================================================================
# STEP 60 — CORRECT OBSERVED / INTERPOLATED FLAGS
# ======================================================================

print("\n" + "=" * 70)
print("STEP 60 — OBSERVED VS INTERPOLATED FLAGS")
print("=" * 70)

interpolation_vars = [
    "income_mean_rm",
    "income_median_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "piped_water_%",
    "sanitation_%",
    "electricity_%"
]

# Reconstruct original observed data from master_panel
for col in interpolation_vars:

    observed_col = f"{col}_observed"

    original_values = (
        master_panel[
            master_panel["year"].isin(dvi_years)
        ]
        .sort_values(["district", "year"])
        [["district", "year", col]]
        .copy()
    )

    # True only when the ORIGINAL source had a value
    original_values[observed_col] = (
        original_values[col].notna()
    )

    # Merge the correct flag into DVI panel
    dvi_panel = dvi_panel.merge(
        original_values[
            ["district", "year", observed_col]
        ],
        on=["district", "year"],
        how="left",
        validate="one_to_one"
    )


# ----------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------

for col in interpolation_vars:

    flag = f"{col}_observed"

    print(
        f"\n{col}"
    )

    print(
        "Observed:",
        dvi_panel[flag].sum()
    )

    print(
        "Interpolated:",
        (~dvi_panel[flag]).sum()
    )


print("\n" + "=" * 70)
print("STEP 60 COMPLETE")
print("=" * 70)


STEP 60 — OBSERVED VS INTERPOLATED FLAGS

income_mean_rm


KeyError: 'income_mean_rm_observed'

In [ ]:
# ======================================================================
# STEP 60 — FIX OBSERVED / INTERPOLATED FLAGS
# ======================================================================

print("\n" + "=" * 70)
print("STEP 60 — FIX OBSERVED / INTERPOLATED FLAGS")
print("=" * 70)

interpolation_vars = [
    "income_mean_rm",
    "income_median_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "piped_water_%",
    "sanitation_%",
    "electricity_%"
]

# Remove the incorrect flags created previously
for col in interpolation_vars:

    flag = f"{col}_observed"

    if flag in dvi_panel.columns:
        dvi_panel = dvi_panel.drop(columns=[flag])


# Recreate TRUE observed-status flags from ORIGINAL master_panel
for col in interpolation_vars:

    flag = f"{col}_observed"

    observed_lookup = (
        master_panel[
            master_panel["year"].isin(dvi_years)
        ][
            ["district", "year", col]
        ]
        .copy()
    )

    observed_lookup[flag] = (
        observed_lookup[col].notna()
    )

    observed_lookup = observed_lookup[
        ["district", "year", flag]
    ]

    dvi_panel = dvi_panel.merge(
        observed_lookup,
        on=["district", "year"],
        how="left",
        validate="one_to_one"
    )


# ----------------------------------------------------------------------
# REPORT
# ----------------------------------------------------------------------

for col in interpolation_vars:

    flag = f"{col}_observed"

    observed = dvi_panel[flag].sum()
    interpolated = (~dvi_panel[flag]).sum()

    print(
        f"\n{col}"
    )

    print(
        "Observed:",
        observed
    )

    print(
        "Interpolated:",
        interpolated
    )


print("\n" + "=" * 70)
print("STEP 60 FIX COMPLETE")
print("=" * 70)


STEP 60 — FIX OBSERVED / INTERPOLATED FLAGS

income_mean_rm
Observed: 33
Interpolated: 33

income_median_rm
Observed: 33
Interpolated: 33

poverty_absolute_pct
Observed: 33
Interpolated: 33

poverty_relative_pct
Observed: 33
Interpolated: 33

piped_water_%
Observed: 33
Interpolated: 33

sanitation_%
Observed: 33
Interpolated: 33

electricity_%
Observed: 33
Interpolated: 33

STEP 60 FIX COMPLETE


In [ ]:
# ======================================================================
# STEP 62 — CPI MONTHLY → ANNUAL
# ======================================================================

print("\n" + "=" * 70)
print("STEP 62 — CPI ANNUAL AGGREGATION")
print("=" * 70)

cpi_annual = inflation.copy()

# Convert date
cpi_annual["date_by_month"] = pd.to_datetime(
    cpi_annual["date_by_month"],
    errors="coerce"
)

# Extract year
cpi_annual["year"] = (
    cpi_annual["date_by_month"]
    .dt.year
)

# Annual average CPI inflation
cpi_annual = (
    cpi_annual
    .groupby(
        ["division", "year"],
        as_index=False
    )
    .agg(
        inflation_yoy_annual=(
            "inflation_yoy",
            "mean"
        ),
        inflation_mom_annual=(
            "inflation_mom",
            "mean"
        )
    )
)

print("\nCPI annual structure:")
print(cpi_annual.head(20))

print("\nRows:", len(cpi_annual))

print(
    "Divisions:",
    cpi_annual["division"].nunique()
)

print(
    "Years:",
    sorted(cpi_annual["year"].unique())
)

print("\nMissing annual YoY:")
print(
    cpi_annual["inflation_yoy_annual"].isna().sum()
)

print("\nMissing annual MoM:")
print(
    cpi_annual["inflation_mom_annual"].isna().sum()
)

print("\n" + "=" * 70)
print("STEP 62 COMPLETE")
print("=" * 70)


STEP 62 — CPI ANNUAL AGGREGATION

CPI annual structure:
   division  year  inflation_yoy_annual  inflation_mom_annual
0        01  2010                   NaN              0.218182
1        01  2011              4.808333              0.450000
2        01  2012              3.183333              0.225000
3        01  2013              4.450000              0.483333
4        01  2014              3.825000              0.125000
5        01  2015              3.441667              0.408333
6        01  2016              3.650000              0.175000
7        01  2017              2.533333              0.258333
8        01  2018              1.125000              0.033333
9        01  2019              0.950000              0.041667
10       01  2020              1.341667              0.183333
11       01  2021              2.216667              0.266667
12       01  2022              4.941667              0.491667
13       01  2023              4.908333              0.191667
14       01  

In [ ]:
# ======================================================================
# STEP 63A — CPI DIVISION DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("STEP 63A — CPI DIVISION DIAGNOSTIC")
print("=" * 70)

print("\n[1] CPI divisions:")
print(sorted(cpi_annual["division"].unique()))

print("\n[2] Division record counts:")
print(
    cpi_annual["division"]
    .value_counts()
    .sort_index()
)

print("\n[3] CPI 2023 values:")
print(
    cpi_annual[
        cpi_annual["year"] == 2023
    ].sort_values("division").to_string(index=False)
)

print("\n" + "=" * 70)


STEP 63A — CPI DIVISION DIAGNOSTIC

[1] CPI divisions:
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', 'overall']

[2] Division record counts:
division
01         17
02         17
03         17
04         17
05         17
06         17
07         17
08         17
09         17
10         17
11         17
12         17
13         17
overall    17
Name: count, dtype: int64

[3] CPI 2023 values:
division  year  inflation_yoy_annual  inflation_mom_annual
      01  2023              4.908333              0.191667
      02  2023              0.350000              0.050000
      03  2023              0.041667              0.000000
      04  2023              1.066667              0.075000
      05  2023              4.141667              0.233333
      06  2023              2.750000              0.316667
      07  2023              1.933333              0.025000
      08  2023             -2.116667             -0.216667
      09  2023              0.825000     

In [ ]:
# ======================================================================
# STEP 63B — OVERALL PAHANG CPI
# ======================================================================

print("\n" + "=" * 70)
print("STEP 63B — OVERALL CPI INFLATION")
print("=" * 70)

cpi_overall = cpi_annual[
    cpi_annual["division"] == "overall"
].copy()

print("\nOverall CPI:")
print(cpi_overall.to_string(index=False))

print("\nYears:")
print(sorted(cpi_overall["year"].unique()))

print("\nMissing annual YoY:")
print(
    cpi_overall["inflation_yoy_annual"].isna().sum()
)

print("\nDuplicate years:")
print(
    cpi_overall["year"].duplicated().sum()
)

print("\n" + "=" * 70)


STEP 63B — OVERALL CPI INFLATION

Overall CPI:
division  year  inflation_yoy_annual  inflation_mom_annual
 overall  2010                   NaN              0.163636
 overall  2011              3.058333              0.250000
 overall  2012              1.850000              0.133333
 overall  2013              2.325000              0.300000
 overall  2014              2.966667              0.166667
 overall  2015              1.625000              0.200000
 overall  2016              1.816667              0.116667
 overall  2017              3.108333              0.241667
 overall  2018              0.600000              0.008333
 overall  2019              0.250000              0.066667
 overall  2020             -1.175000             -0.100000
 overall  2021              2.991667              0.283333
 overall  2022              2.966667              0.300000
 overall  2023              2.683333              0.116667
 overall  2024              2.450000              0.183333
 overall

In [ ]:
# ======================================================================
# STEP 64 — ADD PAHANG OVERALL CPI TO DVI PANEL
# ======================================================================

print("\n" + "=" * 70)
print("STEP 64 — ADD OVERALL PAHANG CPI")
print("=" * 70)

# Keep only the analytical period
cpi_dvi = cpi_overall[
    cpi_overall["year"].isin(dvi_years)
].copy()

# Keep only required columns
cpi_dvi = cpi_dvi[
    [
        "year",
        "inflation_yoy_annual",
        "inflation_mom_annual"
    ]
].copy()

# Rename for final analytical dataset
cpi_dvi = cpi_dvi.rename(
    columns={
        "inflation_yoy_annual": "pahang_inflation_yoy_pct",
        "inflation_mom_annual": "pahang_inflation_mom_pct"
    }
)

# Merge CPI by year only because CPI is Pahang-wide,
# NOT district-specific.
dvi_panel = dvi_panel.merge(
    cpi_dvi,
    on="year",
    how="left",
    validate="many_to_one"
)

print("\nCPI rows:", len(cpi_dvi))

print("\nCPI years:")
print(sorted(cpi_dvi["year"].unique()))

print("\nDVI panel shape:")
print(dvi_panel.shape)

print("\nCPI missing values:")
print(
    dvi_panel[
        [
            "pahang_inflation_yoy_pct",
            "pahang_inflation_mom_pct"
        ]
    ].isna().sum()
)

print("\nCPI values in DVI:")
print(
    dvi_panel[
        [
            "year",
            "pahang_inflation_yoy_pct",
            "pahang_inflation_mom_pct"
        ]
    ]
    .drop_duplicates()
    .sort_values("year")
)

print("\n" + "=" * 70)
print("STEP 64 COMPLETE")
print("=" * 70)


STEP 64 — ADD OVERALL PAHANG CPI

CPI rows: 6

CPI years:
[np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

DVI panel shape:
(66, 44)

CPI missing values:
pahang_inflation_yoy_pct    0
pahang_inflation_mom_pct    0
dtype: int64

CPI values in DVI:
   year  pahang_inflation_yoy_pct  pahang_inflation_mom_pct
0  2019                  0.250000                  0.066667
1  2020                 -1.175000                 -0.100000
2  2021                  2.991667                  0.283333
3  2022                  2.966667                  0.300000
4  2023                  2.683333                  0.116667
5  2024                  2.450000                  0.183333

STEP 64 COMPLETE


In [ ]:
# ======================================================================
# STEP 65 — FINAL ANALYTICAL READINESS AUDIT
# ======================================================================

print("\n" + "=" * 70)
print("STEP 65 — FINAL ANALYTICAL READINESS AUDIT")
print("=" * 70)


# ----------------------------------------------------------------------
# 65.1 — STRUCTURE
# ----------------------------------------------------------------------

print("\n[65.1] STRUCTURE")

print("Rows:", len(dvi_panel))
print("Columns:", len(dvi_panel.columns))

print("Districts:",
      dvi_panel["district"].nunique())

print("Years:",
      sorted(dvi_panel["year"].unique()))


# ----------------------------------------------------------------------
# 65.2 — DISTRICT-YEAR UNIQUENESS
# ----------------------------------------------------------------------

print("\n[65.2] DISTRICT-YEAR UNIQUENESS")

duplicates = dvi_panel.duplicated(
    subset=["district", "year"]
).sum()

print(
    "Duplicate district-year records:",
    duplicates
)


# ----------------------------------------------------------------------
# 65.3 — MISSING VALUES
# ----------------------------------------------------------------------

print("\n[65.3] MISSING VALUES")

missing_final = (
    dvi_panel.isna()
    .sum()
    .sort_values(ascending=False)
)

print(
    missing_final[
        missing_final > 0
    ]
)


# ----------------------------------------------------------------------
# 65.4 — CORE DVI VARIABLES
# ----------------------------------------------------------------------

core_dvi_vars = [
    "labour_force_000",
    "employed_000",
    "unemployed_000",
    "participation_rate_pct",
    "unemployment_rate_pct",
    "employment_pop_ratio_pct",
    "income_mean_rm",
    "income_median_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "population_000",
    "piped_water_%",
    "sanitation_%",
    "electricity_%",
    "students_total",
    "pahang_inflation_yoy_pct"
]

print("\n[65.4] CORE DVI VARIABLES")

print(
    "Missing values in core variables:"
)

print(
    dvi_panel[core_dvi_vars]
    .isna()
    .sum()
)


# ----------------------------------------------------------------------
# 65.5 — DATA TYPES
# ----------------------------------------------------------------------

print("\n[65.5] DATA TYPES")

print(
    dvi_panel[core_dvi_vars]
    .dtypes
)


# ----------------------------------------------------------------------
# 65.6 — BASIC RANGE CHECKS
# ----------------------------------------------------------------------

print("\n[65.6] RANGE CHECKS")

range_checks = {
    "poverty_absolute_pct": (0, 100),
    "poverty_relative_pct": (0, 100),
    "piped_water_%": (0, 100),
    "sanitation_%": (0, 100),
    "electricity_%": (0, 100),
    "participation_rate_pct": (0, 100),
    "unemployment_rate_pct": (0, 100),
    "employment_pop_ratio_pct": (0, 100)
}

for col, (lower, upper) in range_checks.items():

    below = (
        dvi_panel[col] < lower
    ).sum()

    above = (
        dvi_panel[col] > upper
    ).sum()

    print(
        f"{col:35s} "
        f"below={below}, above={above}"
    )


# ----------------------------------------------------------------------
# 65.7 — FINAL OBSERVED / INTERPOLATED COUNTS
# ----------------------------------------------------------------------

print("\n[65.7] OBSERVED VS INTERPOLATED")

for col in [
    "income_mean_rm",
    "income_median_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "piped_water_%",
    "sanitation_%",
    "electricity_%"
]:

    flag = f"{col}_observed"

    print(
        f"{col:30s} "
        f"observed={dvi_panel[flag].sum():2d} "
        f"interpolated={(~dvi_panel[flag]).sum():2d}"
    )


# ----------------------------------------------------------------------
# 65.8 — FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL ANALYTICAL READINESS CHECK")
print("=" * 70)


STEP 65 — FINAL ANALYTICAL READINESS AUDIT

[65.1] STRUCTURE
Rows: 66
Columns: 44
Districts: 11
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

[65.2] DISTRICT-YEAR UNIQUENESS
Duplicate district-year records: 0

[65.3] MISSING VALUES
total_employment                55
avg_annual_salary_rm            55
crop_production_total_tonnes    55
population_000                  11
dtype: int64

[65.4] CORE DVI VARIABLES
Missing values in core variables:
labour_force_000             0
employed_000                 0
unemployed_000               0
participation_rate_pct       0
unemployment_rate_pct        0
employment_pop_ratio_pct     0
income_mean_rm               0
income_median_rm             0
poverty_absolute_pct         0
poverty_relative_pct         0
population_000              11
piped_water_%                0
sanitation_%                 0
electricity_%                0
students_total               0
pahang_inflation_yoy_pct     

In [ ]:
# ======================================================================
# STEP 66 — FINAL CORE DVI PANEL
# ======================================================================

print("\n" + "=" * 70)
print("STEP 66 — FINAL CORE DVI PANEL")
print("=" * 70)

# ----------------------------------------------------------------------
# 66.1 — Restrict to 2020–2024
# ----------------------------------------------------------------------

dvi_core = dvi_panel[
    dvi_panel["year"].between(2020, 2024)
].copy()

dvi_core = dvi_core.sort_values(
    ["district", "year"]
).reset_index(drop=True)


# ----------------------------------------------------------------------
# 66.2 — Define CORE DVI variables
# ----------------------------------------------------------------------

core_dvi_vars = [
    "labour_force_000",
    "employed_000",
    "unemployed_000",
    "outside_lf_000",
    "participation_rate_pct",
    "unemployment_rate_pct",
    "employment_pop_ratio_pct",

    "income_mean_rm",
    "income_median_rm",

    "poverty_absolute_pct",
    "poverty_relative_pct",

    "population_000",

    "piped_water_%",
    "sanitation_%",
    "electricity_%",

    "students_total",

    "pahang_inflation_yoy_pct",
    "pahang_inflation_mom_pct"
]


# ----------------------------------------------------------------------
# 66.3 — Check core missingness
# ----------------------------------------------------------------------

print("\n[66.3] CORE MISSINGNESS")

core_missing = (
    dvi_core[core_dvi_vars]
    .isna()
    .sum()
)

print(core_missing)


# ----------------------------------------------------------------------
# 66.4 — Check district-year uniqueness
# ----------------------------------------------------------------------

print("\n[66.4] DISTRICT-YEAR UNIQUENESS")

print(
    "Duplicate records:",
    dvi_core.duplicated(
        subset=["district", "year"]
    ).sum()
)


# ----------------------------------------------------------------------
# 66.5 — Coverage
# ----------------------------------------------------------------------

print("\n[66.5] COVERAGE")

print("Rows:", len(dvi_core))
print("Districts:", dvi_core["district"].nunique())
print("Years:", sorted(dvi_core["year"].unique()))

print("\nRows per year:")
print(
    dvi_core.groupby("year").size()
)


# ----------------------------------------------------------------------
# 66.6 — Supplementary variables
# ----------------------------------------------------------------------

supplementary_vars = [
    "crop_production_total_tonnes",
    "total_employment",
    "avg_annual_salary_rm"
]

print("\n[66.6] SUPPLEMENTARY 2023 VARIABLES")

print(
    dvi_core[
        supplementary_vars
    ].notna().sum()
)

print(
    "\nThese variables are retained as "
    "supplementary 2023 agricultural/livelihood indicators."
)


# ----------------------------------------------------------------------
# 66.7 — FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CORE DVI PANEL CHECK")
print("=" * 70)

if (
    len(dvi_core) == 55
    and dvi_core["district"].nunique() == 11
    and dvi_core["year"].nunique() == 5
    and dvi_core[core_dvi_vars].isna().sum().sum() == 0
    and dvi_core.duplicated(
        subset=["district", "year"]
    ).sum() == 0
):

    print("STATUS: PASSED")
    print("Core DVI panel is analytically ready.")

else:

    print("STATUS: REVIEW REQUIRED")

print("=" * 70)


STEP 66 — FINAL CORE DVI PANEL

[66.3] CORE MISSINGNESS
labour_force_000            0
employed_000                0
unemployed_000              0
outside_lf_000              0
participation_rate_pct      0
unemployment_rate_pct       0
employment_pop_ratio_pct    0
income_mean_rm              0
income_median_rm            0
poverty_absolute_pct        0
poverty_relative_pct        0
population_000              0
piped_water_%               0
sanitation_%                0
electricity_%               0
students_total              0
pahang_inflation_yoy_pct    0
pahang_inflation_mom_pct    0
dtype: int64

[66.4] DISTRICT-YEAR UNIQUENESS
Duplicate records: 0

[66.5] COVERAGE
Rows: 55
Districts: 11
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Rows per year:
year
2020    11
2021    11
2022    11
2023    11
2024    11
dtype: int64

[66.6] SUPPLEMENTARY 2023 VARIABLES
crop_production_total_tonnes    11
total_employment                11
avg_annual_s

In [ ]:
# ======================================================================
# STEP 67 — FREEZE FINAL ANALYTICAL DATASET
# ======================================================================

print("\n" + "=" * 70)
print("STEP 67 — FREEZE FINAL ANALYTICAL DATASET")
print("=" * 70)

# ----------------------------------------------------------------------
# Final sort
# ----------------------------------------------------------------------

dvi_core = dvi_core.sort_values(
    ["district", "year"]
).reset_index(drop=True)


# ----------------------------------------------------------------------
# Save CSV
# ----------------------------------------------------------------------

dvi_core.to_csv(
    "DVI_core_2020_2024.csv",
    index=False
)

print("\nSaved:")
print("DVI_core_2020_2024.csv")


# ----------------------------------------------------------------------
# Final shape
# ----------------------------------------------------------------------

print("\nShape:")
print(dvi_core.shape)


# ----------------------------------------------------------------------
# Preview
# ----------------------------------------------------------------------

print("\nFirst 10 rows:")
print(
    dvi_core[
        ["district", "year"]
        + core_dvi_vars
    ].head(10).to_string(index=False)
)


print("\n" + "=" * 70)
print("ANALYTICAL DATASET FROZEN")
print("=" * 70)


STEP 67 — FREEZE FINAL ANALYTICAL DATASET

Saved:
DVI_core_2020_2024.csv

Shape:
(55, 44)

First 10 rows:
district  year  labour_force_000  employed_000  unemployed_000  outside_lf_000  participation_rate_pct  unemployment_rate_pct  employment_pop_ratio_pct  income_mean_rm  income_median_rm  poverty_absolute_pct  poverty_relative_pct  population_000  piped_water_%  sanitation_%  electricity_%  students_total  pahang_inflation_yoy_pct  pahang_inflation_mom_pct
 Bentong  2020              52.5          50.7             1.8            26.8                    66.2                    3.4                      63.9     5387.666667       4377.000000              3.133333                  2.60           933.3      99.300000         100.0          100.0           33876                 -1.175000                 -0.100000
 Bentong  2021              51.5          50.0             1.4            27.5                    65.2                    2.8                      63.3     5475.333333       453

# Check correlations before selecting indicators

In [ ]:
# ======================================================================
# STEP 68A — DVI INDICATOR CORRELATION CHECK
# ======================================================================

print("\n" + "=" * 70)
print("STEP 68A — DVI INDICATOR CORRELATION CHECK")
print("=" * 70)

candidate_vars = [
    "labour_force_000",
    "employed_000",
    "unemployed_000",
    "outside_lf_000",
    "participation_rate_pct",
    "unemployment_rate_pct",
    "employment_pop_ratio_pct",
    "income_mean_rm",
    "income_median_rm",
    "poverty_absolute_pct",
    "poverty_relative_pct",
    "piped_water_%",
    "sanitation_%",
    "electricity_%",
    "students_total",
    "pahang_inflation_yoy_pct"
]

corr_matrix = (
    dvi_core[candidate_vars]
    .corr(method="spearman")
    .round(2)
)

print(corr_matrix.to_string())

print("\n" + "=" * 70)
print("STEP 68A COMPLETE")
print("=" * 70)


STEP 68A — DVI INDICATOR CORRELATION CHECK
                          labour_force_000  employed_000  unemployed_000  outside_lf_000  participation_rate_pct  unemployment_rate_pct  employment_pop_ratio_pct  income_mean_rm  income_median_rm  poverty_absolute_pct  poverty_relative_pct  piped_water_%  sanitation_%  electricity_%  students_total  pahang_inflation_yoy_pct
labour_force_000                      1.00          1.00            0.71            0.55                    0.36                  -0.03                      0.38            0.30              0.31                 -0.09                  0.09           0.51          0.41           0.35            0.83                     -0.09
employed_000                          1.00          1.00            0.68            0.57                    0.35                  -0.07                      0.37            0.31              0.33                 -0.08                  0.08           0.51          0.39           0.34            0.85     

In [ ]:
# ======================================================================
# STEP 68 — FINAL PHASE 2 PREPROCESSING AUDIT
# ======================================================================

print("\n" + "=" * 70)
print("FINAL PHASE 2 — DATA PREPROCESSING AUDIT")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. STRUCTURE
# ----------------------------------------------------------------------

print("\n[1] MASTER DATASET STRUCTURE")

print("Rows:", len(dvi_core))
print("Columns:", len(dvi_core.columns))

print(
    "Districts:",
    dvi_core["district"].nunique()
)

print(
    "Years:",
    sorted(dvi_core["year"].unique())
)


# ----------------------------------------------------------------------
# 2. DISTRICT-YEAR DUPLICATES
# ----------------------------------------------------------------------

print("\n[2] DUPLICATE DISTRICT-YEAR CHECK")

print(
    "Duplicate district-year records:",
    dvi_core.duplicated(
        subset=["district", "year"]
    ).sum()
)


# ----------------------------------------------------------------------
# 3. MISSING VALUES
# ----------------------------------------------------------------------

print("\n[3] MISSING VALUES")

missing = dvi_core.isna().sum()

print(
    missing[missing > 0]
)

if missing.sum() == 0:
    print("No missing values in core analytical panel.")


# ----------------------------------------------------------------------
# 4. DATA TYPES
# ----------------------------------------------------------------------

print("\n[4] DATA TYPES")

print(
    dvi_core.dtypes
)


# ----------------------------------------------------------------------
# 5. NUMERIC VARIABLES
# ----------------------------------------------------------------------

print("\n[5] NON-NUMERIC CORE VARIABLES")

numeric_check = dvi_core[
    core_dvi_vars
].select_dtypes(
    exclude="number"
).columns.tolist()

print(numeric_check)

if len(numeric_check) == 0:
    print("All core analytical variables are numeric.")


# ----------------------------------------------------------------------
# 6. RANGE CHECKS
# ----------------------------------------------------------------------

print("\n[6] RANGE CHECKS")

range_checks = {
    "participation_rate_pct": (0, 100),
    "unemployment_rate_pct": (0, 100),
    "employment_pop_ratio_pct": (0, 100),
    "poverty_absolute_pct": (0, 100),
    "poverty_relative_pct": (0, 100),
    "piped_water_%": (0, 100),
    "sanitation_%": (0, 100),
    "electricity_%": (0, 100)
}

for col, (lower, upper) in range_checks.items():

    below = (
        dvi_core[col] < lower
    ).sum()

    above = (
        dvi_core[col] > upper
    ).sum()

    print(
        f"{col:35s} "
        f"below={below}, above={above}"
    )


# ----------------------------------------------------------------------
# 7. DISTRICT NAMES
# ----------------------------------------------------------------------

print("\n[7] FINAL DISTRICT NAMES")

print(
    sorted(
        dvi_core["district"].unique()
    )
)


# ----------------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 2 PREPROCESSING COMPLETE")
print("=" * 70)

print("""
Phase 2 requirements completed:
✓ Raw data processing
✓ District crosswalk
✓ District standardization
✓ Date standardization
✓ Numeric conversion
✓ Missingness analysis
✓ Duplicate checks
✓ Unit/range verification
✓ Cross-dataset validation
✓ Temporal coverage validation
✓ District-year aggregation
✓ Integrated master panel
""")

print("=" * 70)


FINAL PHASE 2 — DATA PREPROCESSING AUDIT

[1] MASTER DATASET STRUCTURE
Rows: 55
Columns: 44
Districts: 11
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

[2] DUPLICATE DISTRICT-YEAR CHECK
Duplicate district-year records: 0

[3] MISSING VALUES
crop_production_total_tonnes    44
total_employment                44
avg_annual_salary_rm            44
dtype: int64

[4] DATA TYPES
district                               str
year                                 int64
labour_force_000                   float64
employed_000                       float64
unemployed_000                     float64
outside_lf_000                     float64
participation_rate_pct             float64
unemployment_rate_pct              float64
employment_pop_ratio_pct           float64
income_mean_rm                     float64
income_median_rm                   float64
poverty_absolute_pct               float64
poverty_relative_pct               float64
population_000        